# SAEs Reveal Algorithmic Representations in NARs — Experiments

End-to-end pipeline:
1. **Phase 1**: Train NAR on CLRS-30 (BFS first)
2. **Phase 2**: Collect activations & train SAE
3. **Phase 3**: Feature interpretation & concept correlations

In [ ]:
import sys
sys.version

In [ ]:
!nvidia-smi

In [ ]:
# --- Colab setup (skip if running locally) ---
import os

IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in str(os.environ.get("COLAB_RELEASE_TAG", ""))

if IN_COLAB:
    if not os.path.exists("/content/nar-mechinterp"):
        !git clone https://github.com/aniervs/nar-mechinterp.git
        %cd nar-mechinterp
        # Install salsa-clrs from git first (not on PyPI, pip can't resolve it from pyproject.toml)
        !pip install salsa-clrs@git+https://github.com/jkminder/SALSA-CLRS.git 2>&1 | tail -3
        # Install the project + remaining deps
        !pip install -e . 2>&1 | tail -5
        # Force-reinstall numpy/scipy to avoid Colab's stale pre-installed versions
        !pip install --force-reinstall numpy scipy 2>&1 | tail -3
        print("Colab setup complete! Restarting runtime — re-run this cell after restart.")
        import IPython
        IPython.get_ipython().kernel.do_shutdown(restart=True)
    else:
        %cd /content/nar-mechinterp
        print("Colab setup already done.")

    # Mount Google Drive for persistent storage
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Running locally, skipping Colab setup.")

In [ ]:
import sys
from pathlib import Path

# Handle import paths for both Colab (cwd=repo root) and local (cwd=experiments/)
if Path("../data").exists() and ".." not in sys.path:
    sys.path.insert(0, "..")
elif Path("data").exists() and "." not in sys.path:
    sys.path.insert(0, ".")

# Fix PyG pickling issue on some versions (Colab)
try:
    import torch_geometric.data.data as _pyg_data
    if not hasattr(_pyg_data, 'DataEdgeAttr'):
        _pyg_data.DataEdgeAttr = type('DataEdgeAttr', (), {})
        _pyg_data.DataTensorAttr = type('DataTensorAttr', (), {})
except Exception:
    pass

import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
import numpy as np

from data import (
    get_clrs_dataset,
    get_clrs_dataloader,
    get_algorithm_spec,
    spec_to_model_types,
    batch_to_model_inputs,
)
from models import NARModel
from interp import (
    SparseAutoencoder, BatchTopKSAE, Transcoder,
    SAETrainer, SAEOutput,
    ActivationCollector, make_activation_dataloader,
    FeatureAnalyzer, FeatureAnalysisResult,
)
from interp.concept_labels import collect_concept_labels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

SEED = 42
torch.manual_seed(SEED)

## Configuration

In [ ]:
# --- Experiment configuration ---
ALGORITHM = "bfs"  # Start with BFS (simplest, clearest ground-truth concepts)

# === Scale toggle ===
# Set to True for quick local testing (M1 MacBook), False for full runs on GPU
LOCAL_DEBUG = False

if LOCAL_DEBUG:
    # Small model for quick sanity checks (~30K params)
    HIDDEN_DIM = 32
    NUM_LAYERS = 2
    NUM_HEADS = 4
    NAR_EPOCHS = 20
    TRAIN_SAMPLES = 200
    VAL_SAMPLES = 64
    SAE_STEPS = 5_000
    SAE_NUM_SAMPLES = 500
else:
    # Full-scale model for real experiments (~1.5M params)
    HIDDEN_DIM = 128
    NUM_LAYERS = 4
    NUM_HEADS = 8
    NAR_EPOCHS = 100
    TRAIN_SAMPLES = 1000
    VAL_SAMPLES = 128
    SAE_STEPS = 50_000
    SAE_NUM_SAMPLES = 10_000

PROCESSOR_TYPE = "mpnn"

# Training (shared)
NAR_BATCH_SIZE = 32
NAR_LR = 1e-3
NUM_NODES = 16
EDGE_PROB = 0.2

# SAE hyperparameters
SAE_TYPE = "batchtopk"
EXPANSION_FACTOR = 4     # dict_size = HIDDEN_DIM * EXPANSION_FACTOR (reduced from 8 to cut dead features)
SAE_K = 16 if not LOCAL_DEBUG else 8  # fewer active features → sparser, more monosemantic
SAE_LR = 3e-4
SAE_BATCH_SIZE = 256

# Paths — use Google Drive on Colab for persistence
REPO_ROOT = Path("..") if Path("../data").exists() else Path(".")

if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/nar-mechinterp")
    CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / ALGORITHM
    RESULTS_DIR = DRIVE_ROOT / "results" / "sae" / ALGORITHM / SAE_TYPE
else:
    CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / ALGORITHM
    RESULTS_DIR = REPO_ROOT / "results" / "sae" / ALGORITHM / SAE_TYPE

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Mode: {'LOCAL DEBUG' if LOCAL_DEBUG else 'FULL SCALE'}")
print(f"Algorithm: {ALGORITHM}")
print(f"Saving to: {CHECKPOINT_DIR}")
print(f"NAR: hidden_dim={HIDDEN_DIM}, layers={NUM_LAYERS}, heads={NUM_HEADS}, processor={PROCESSOR_TYPE}")
print(f"SAE: type={SAE_TYPE}, expansion={EXPANSION_FACTOR}x, dict_size={HIDDEN_DIM * EXPANSION_FACTOR}, k={SAE_K}")
print(f"Training: {NAR_EPOCHS} epochs, {TRAIN_SAMPLES} samples")

In [ ]:
# ── Optional clean run ────────────────────────────────────────────────────────
# Set CLEAN_RUN = True to delete all cached outputs and start from scratch.
# Checkpoints, activations, SAE weights, probe results — everything gets wiped.
CLEAN_RUN = False

if CLEAN_RUN:
    import shutil

    _cr_root = RESULTS_DIR.parent.parent  # e.g. results/sae/

    dirs_to_delete = [
        CHECKPOINT_DIR,
        RESULTS_DIR.parent,                                          # results/sae/<algo>/
        *[_cr_root / f"sae_layer{i}" for i in range(NUM_LAYERS)],
        _cr_root / "sae_layer0_small",
        _cr_root / "sae_frontier",
        _cr_root / "linear_probes",
        _cr_root / "sweep",
    ]
    loose_files = [
        "per_step_analysis.pt",
        "frontier_activations.pt",
        "frontier_enrichment.pt",
        "summary_sae_vs_probe.png",
    ]

    for d in dirs_to_delete:
        if d.exists():
            shutil.rmtree(d)
            print(f"Deleted {d}")

    for fname in loose_files:
        f = _cr_root / fname
        if f.exists():
            f.unlink()
            print(f"Deleted {f}")

    print("\nAll outputs cleared. Re-run from the top.")

---
## Phase 1: Train NAR on CLRS-30

Train an MPNN-based NAR model on BFS. The model learns to predict BFS outputs (reachability) with hint supervision at each message-passing step.

### 1.1 Load CLRS-30 dataset

In [ ]:
# Load dataset and extract algorithm spec
ds = get_clrs_dataset(
    ALGORITHM, split="train",
    num_samples=TRAIN_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
)
spec = get_algorithm_spec(ALGORITHM, dataset=ds)
output_types, hint_types = spec_to_model_types(spec)

print(f"Output types: {output_types}")
print(f"Hint types: {hint_types}")

# Create dataloaders
train_loader = get_clrs_dataloader(
    ALGORITHM, "train",
    batch_size=NAR_BATCH_SIZE,
    num_samples=TRAIN_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED,
)
val_loader = get_clrs_dataloader(
    ALGORITHM, "val",
    batch_size=NAR_BATCH_SIZE,
    num_samples=VAL_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED + 1,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

### 1.2 Train NAR model

In [ ]:
model = NARModel(
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    processor_type=PROCESSOR_TYPE,
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"NAR model parameters: {num_params:,}")

In [ ]:
def train_epoch(model, loader, optimizer, spec, output_types, hint_types, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        inputs, outputs, hints = batch_to_model_inputs(batch, spec, device)
        optimizer.zero_grad()
        output = model(
            inputs=inputs, hints=hints, outputs=outputs,
            output_types=output_types, hint_types=hint_types,
            num_steps=batch.lengths.max().item(),
        )
        output.total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += output.total_loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, spec, output_types, hint_types, device):
    model.eval()
    total_loss = 0
    total_acc = 0
    count = 0
    for batch in loader:
        batch = batch.to(device)
        inputs, outputs, hints = batch_to_model_inputs(batch, spec, device)
        output = model(
            inputs=inputs, outputs=outputs,
            output_types=output_types, hint_types=hint_types,
            num_steps=batch.lengths.max().item(),
        )
        total_loss += output.total_loss.item()
        for name, pred in output.predictions.items():
            if name in outputs:
                target = outputs[name]
                if output_types.get(name) == 'node_mask':
                    pred_bin = torch.sigmoid(pred[:, :target.shape[-1]]) > 0.5
                    acc = (pred_bin == target.bool()).float().mean()
                elif output_types.get(name) == 'node_pointer':
                    pred_idx = pred.argmax(-1)
                    tgt_idx = target.long() if target.dim() < pred.dim() else target.argmax(-1)
                    acc = (pred_idx == tgt_idx).float().mean()
                else:
                    acc = torch.tensor(0.0)
                total_acc += acc.item()
                count += 1
    return total_loss / max(len(loader), 1), total_acc / max(count, 1)

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=NAR_LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NAR_EPOCHS)

train_losses, val_losses, val_accs = [], [], []
best_loss = float('inf')

for epoch in range(NAR_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, spec, output_types, hint_types, device)
    val_loss, val_acc = validate(model, val_loader, spec, output_types, hint_types, device)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'config': {
                'hidden_dim': HIDDEN_DIM, 'num_layers': NUM_LAYERS,
                'num_heads': NUM_HEADS, 'processor_type': PROCESSOR_TYPE,
            },
        }, CHECKPOINT_DIR / "best.pt")

    print(f"Epoch {epoch+1}/{NAR_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Acc: {val_acc:.4f}")

torch.save({'model_state_dict': model.state_dict()}, CHECKPOINT_DIR / "final.pt")

# Save training history
torch.save({
    'train_losses': train_losses,
    'val_losses': val_losses,
    'val_accs': val_accs,
}, CHECKPOINT_DIR / "training_history.pt")

print(f"\nBest val loss: {best_loss:.4f}")
print(f"Saved training history to {CHECKPOINT_DIR / 'training_history.pt'}")

### 1.3 Training curves

In [ ]:
if 'train_losses' not in dir():
    _hist = torch.load(CHECKPOINT_DIR / "training_history.pt", weights_only=True)
    train_losses = _hist['train_losses']
    val_losses   = _hist['val_losses']
    val_accs     = _hist['val_accs']
    print("Loaded training history from disk")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label="Train")
ax1.plot(val_losses, label="Val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title(f"NAR Training — {ALGORITHM.upper()}")
ax1.legend(); ax1.set_yscale("log")
ax2.plot(val_accs)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.set_title("Validation Accuracy"); ax2.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

---
## Phase 2: Collect Activations & Train SAE

Collect processor hidden states from the trained NAR, then train a BatchTopK SAE to discover monosemantic features. Each (node, message-passing step) pair is treated as an independent sample.

### 2.1 Load best NAR checkpoint

In [ ]:
# Load best checkpoint
ckpt = torch.load(CHECKPOINT_DIR / "best.pt", map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}, val_acc={ckpt['val_acc']:.4f})")

### 2.2 Collect processor activations

In [ ]:
_act_path = RESULTS_DIR / "activations.pt"
if 'activations' not in dir() and _act_path.exists():
    activations = torch.load(_act_path)
    print(f"Loaded activations from disk: {activations.shape}")
else:
    act_loader = get_clrs_dataloader(
        ALGORITHM, "train",
        batch_size=32, num_samples=SAE_NUM_SAMPLES,
        num_nodes=NUM_NODES, edge_probability=EDGE_PROB,
        seed=SEED + 100, shuffle=False,
    )
    collector = ActivationCollector(model, spec=spec, device=device)
    activations = collector.collect(act_loader, output_types=output_types)
    print(f"Collected activations: {activations.shape}")
    torch.save(activations, _act_path)
    print(f"Saved to {_act_path}")

In [ ]:
activations = torch.load(RESULTS_DIR / "activations.pt")

### 2.3 Collect concept labels

Extract ground-truth algorithmic concept labels (is_visited, is_frontier, is_source, etc.) aligned with the activation vectors.

In [ ]:
_lbl_path = RESULTS_DIR / "concept_labels.pt"
if _lbl_path.exists():
    _saved = torch.load(_lbl_path, weights_only=True)
    print(f"Loaded concept labels from disk: {list(_saved['labels'].keys())}")
else:
    label_loader = get_clrs_dataloader(
        ALGORITHM, "train",
        batch_size=32, num_samples=SAE_NUM_SAMPLES,
        num_nodes=NUM_NODES, edge_probability=EDGE_PROB,
        seed=SEED + 100, shuffle=False,
    )
    concept_result = collect_concept_labels(label_loader, spec, ALGORITHM)
    print(f"Concept labels collected: {concept_result.num_samples} samples")
    print(f"Concepts: {list(concept_result.labels.keys())}")
    torch.save({
        "labels": concept_result.labels,
        "descriptions": concept_result.concept_descriptions,
        "algorithm": concept_result.algorithm,
    }, _lbl_path)
    print(f"Saved to {_lbl_path}")

### 2.4 Train BatchTopK SAE

In [ ]:
dict_size = HIDDEN_DIM * EXPANSION_FACTOR

sae = BatchTopKSAE(
    input_dim=HIDDEN_DIM,
    dict_size=dict_size,
    k=SAE_K,
).to(device)

print(f"BatchTopK SAE: input_dim={HIDDEN_DIM}, dict_size={dict_size}, k={SAE_K}")
print(f"SAE parameters: {sum(p.numel() for p in sae.parameters()):,}")

In [ ]:
trainer = SAETrainer(sae, lr=SAE_LR, resample_dead_every=0)
sae_loader = make_activation_dataloader(activations, batch_size=SAE_BATCH_SIZE)

sae_losses = []
step = 0

while step < SAE_STEPS:
    for (batch,) in sae_loader:
        if step >= SAE_STEPS:
            break
        batch = batch.to(device)
        output = trainer.train_step(batch)

        if step % 500 == 0:
            sae_losses.append({
                'step': step,
                'loss': output.loss.item(),
                'recon': output.reconstruction_loss.item(),
                'l0': output.l0.item(),
            })
            print(f"Step {step}/{SAE_STEPS} | loss={output.loss.item():.4f} | recon={output.reconstruction_loss.item():.4f} | L0={output.l0.item():.1f}")

        # Save checkpoint every 10K steps
        if step > 0 and step % 10_000 == 0:
            torch.save({
                "state_dict": sae.state_dict(),
                "config": sae.get_config(),
                "sae_type": SAE_TYPE,
                "step": step,
            }, RESULTS_DIR / f"sae_step{step}.pt")
            print(f"  [checkpoint saved: step {step}]")

        step += 1

print(f"\nFinal training stats: {trainer.get_training_stats()}")

# Save SAE training losses
torch.save(sae_losses, RESULTS_DIR / "sae_losses.pt")
print(f"Saved training losses to {RESULTS_DIR / 'sae_losses.pt'}")

In [ ]:
# Save SAE checkpoint
torch.save({
    "state_dict": sae.state_dict(),
    "config": sae.get_config(),
    "sae_type": SAE_TYPE,
}, RESULTS_DIR / "sae.pt")
print(f"Saved SAE to {RESULTS_DIR / 'sae.pt'}")

In [ ]:
ckpt = torch.load(RESULTS_DIR / "sae.pt", map_location=device, weights_only=True)
sae.load_state_dict(ckpt["state_dict"])
sae.eval()
print(f"Loaded SAE from {RESULTS_DIR / 'sae.pt'}")

### 2.5 SAE training curves

In [ ]:
if 'sae_losses' not in dir():
    sae_losses = torch.load(RESULTS_DIR / "sae_losses.pt", weights_only=True)
    print("Loaded sae_losses from disk")

steps = [d['step'] for d in sae_losses]
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
ax1.plot(steps, [d['loss']  for d in sae_losses]); ax1.set_title("SAE Total Loss")
ax2.plot(steps, [d['recon'] for d in sae_losses]); ax2.set_title("SAE Reconstruction Loss")
ax3.plot(steps, [d['l0']    for d in sae_losses]); ax3.set_title("SAE Sparsity (L0)")
ax3.axhline(y=SAE_K, color='r', linestyle='--', alpha=0.5, label=f"target k={SAE_K}")
for ax in (ax1, ax2, ax3):
    ax.set_xlabel("Step")
ax3.legend()
plt.tight_layout()
plt.show()

---
## Phase 3: Feature Interpretation

Analyze SAE features: basic statistics, reconstruction quality, and — most importantly — correlations with ground-truth algorithmic concepts (is_visited, is_frontier, etc.).

### 3.1 Reconstruction quality & dead features

In [ ]:
sae.eval()
analyzer = FeatureAnalyzer(sae)

with torch.no_grad():
    sample = activations[:10_000].to(device)
    # Evaluate in same batch size as training to match BatchTopK behavior
    recon_list = []
    for i in range(0, len(sample), SAE_BATCH_SIZE):
        batch = sample[i:i+SAE_BATCH_SIZE]
        out = sae(batch)
        recon_list.append(out.reconstructed)
    reconstructed = torch.cat(recon_list)

    recon_mse = F.mse_loss(reconstructed, sample).item()
    total_var = sample.var(dim=0).sum()
    residual_var = (sample - reconstructed).var(dim=0).sum()
    fve = (1 - residual_var / total_var).item()

print(f"Reconstruction MSE: {recon_mse:.6f}")
print(f"Fraction of variance explained: {fve:.4f}")

# Dead features (also batched)
with torch.no_grad():
    all_features = []
    for i in range(0, len(sample), SAE_BATCH_SIZE):
        batch = sample[i:i+SAE_BATCH_SIZE]
        out = sae(batch)
        all_features.append(out.features)
    features = torch.cat(all_features)
    num_dead = (features.sum(dim=0) == 0).sum().item()
print(f"Dead features: {num_dead}/{dict_size} ({100*num_dead/dict_size:.1f}%)")

l0_per_sample = (features > 0).float().sum(dim=-1)
print(f"L0 (avg active features): {l0_per_sample.mean():.1f} +/- {l0_per_sample.std():.1f}")
print(f"L0 range: [{l0_per_sample.min():.0f}, {l0_per_sample.max():.0f}]")


### 3.2 Feature statistics

In [ ]:
stats = analyzer.compute_feature_stats(activations.to(device))
stats_sorted = sorted(stats, key=lambda s: s.activation_frequency, reverse=True)

# Feature frequency distribution
freqs = [s.activation_frequency for s in stats]
active_freqs = [f for f in freqs if f > 0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(freqs, bins=50, edgecolor='black', alpha=0.7)
ax1.set_xlabel("Activation Frequency")
ax1.set_ylabel("Count")
ax1.set_title("Feature Activation Frequency Distribution")
ax1.axvline(x=0, color='r', linestyle='--', alpha=0.5)

if active_freqs:
    ax2.hist(active_freqs, bins=50, edgecolor='black', alpha=0.7)
    ax2.set_xlabel("Activation Frequency")
    ax2.set_ylabel("Count")
    ax2.set_title("Active Features Only")

plt.tight_layout()
plt.show()

# Top-20 most active features
print("Top-20 most active features:")
for s in stats_sorted[:20]:
    print(f"  Feature {s.feature_idx:4d}: freq={s.activation_frequency:.4f}, "
          f"mean_act={s.mean_activation:.4f}, max_act={s.max_activation:.4f}")

### 3.3 Concept correlations — the core result

For each SAE feature, compute Pearson correlation with ground-truth BFS concepts:
- **is_visited**: node has been reached
- **is_frontier**: node just entered the wavefront this step
- **is_source**: source node
- **is_unvisited**: node not yet reached

In [ ]:
_corr_path.exists()

In [ ]:
_corr_path = RESULTS_DIR / "correlation_results.pt"
if _corr_path.exists() and 'concept_matrix' not in dir():
    _r = torch.load(_corr_path, weights_only=False)
    concept_matrix = _r["concept_matrix"]
    concept_names  = _r["concept_names"]
    mono           = _r["mono"]
    dead           = _r["dead"]
    features       = _r["features"]
    print(f"Loaded correlation results: {len(mono)} mono, {len(dead)} dead")
else:
    # Load concept labels
    concept_data = torch.load(RESULTS_DIR / "concept_labels.pt", map_location=device, weights_only=True)
    concept_labels = concept_data["labels"]
    
    # Drop redundant concepts (is_unvisited = 1 - is_visited)
    concept_labels.pop("is_unvisited", None)
    
    # Align dimensions
    N_CORR = 10_000
    min_n = min(N_CORR, activations.shape[0], min(v.shape[0] for v in concept_labels.values()))
    
    concept_names = list(concept_labels.keys())
    labels = torch.stack([concept_labels[name][:min_n].float().cpu() for name in concept_names], dim=1)
    
    print(f"Samples: {min_n:,}")
    print(f"Concepts: {concept_names}")
    
    # Batch-encode features on GPU, collect on CPU
    torch.cuda.empty_cache()
    all_features = []
    with torch.no_grad():
        for i in range(0, min_n, SAE_BATCH_SIZE):
            batch = activations[i:i+SAE_BATCH_SIZE].to(device)
            out = sae(batch)
            all_features.append(out.features.cpu())
            del out, batch
            torch.cuda.empty_cache()
    features = torch.cat(all_features).float()
    features = features[:min_n]
    del all_features
    
    # Correlations on CPU
    feat_centered = features - features.mean(dim=0, keepdim=True)
    label_centered = labels - labels.mean(dim=0, keepdim=True)
    feat_norm = feat_centered / feat_centered.std(dim=0, keepdim=True).clamp(min=1e-8)
    label_norm = label_centered / label_centered.std(dim=0, keepdim=True).clamp(min=1e-8)
    concept_matrix = (feat_norm.T @ label_norm) / min_n
    del feat_centered, label_centered, feat_norm, label_norm
    
    # Identify monosemantic features (relaxed: allow anti-correlation with complement)
    mono = []
    for i in range(dict_size):
        corrs = concept_matrix[i].abs()
        if corrs.max() > 0.5 and (corrs > 0.3).sum() <= 2:
            mono.append(i)
    
    dead = (features.sum(dim=0) == 0).nonzero(as_tuple=True)[0].tolist()
    
    print(f"\nMonosemantic features: {len(mono)} / {dict_size}")
    print(f"Dead features: {len(dead)} / {dict_size}")
    
    # Save correlation results
    torch.save({
        "concept_matrix": concept_matrix,
        "concept_names": concept_names,
        "mono": mono,
        "dead": dead,
        "features": features,
    }, RESULTS_DIR / "correlation_results.pt")
    print(f"Saved correlation results to {RESULTS_DIR / 'correlation_results.pt'}")


In [ ]:
# Top correlated features per concept
for j, concept_name in enumerate(concept_names):
    corrs = concept_matrix[:, j]
    top_indices = corrs.abs().topk(5).indices
    print(f"\nConcept '{concept_name}' — top correlated features:")
    for idx in top_indices:
        print(f"  Feature {idx.item():4d}: correlation = {corrs[idx]:+.4f}")


### 3.4 Concept correlation heatmap

In [ ]:
# Heatmap of concept correlations for the top-K most interesting features
cm_np = concept_matrix.cpu().numpy()  # (dict_size, num_concepts)
max_corr_per_feat = np.abs(cm_np).max(axis=1)
top_k_feats = 30
top_feat_idxs = np.argsort(max_corr_per_feat)[-top_k_feats:][::-1]

fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(cm_np[top_feat_idxs], cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)

ax.set_yticks(range(len(top_feat_idxs)))
ax.set_yticklabels([f"Feature {i}" for i in top_feat_idxs])
ax.set_xticks(range(len(concept_names)))
ax.set_xticklabels(concept_names, rotation=45, ha='right')
ax.set_title(f"Top-{top_k_feats} Features x BFS Concepts — Pearson Correlation")

plt.colorbar(im, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()


### 3.5 Monosemanticity analysis

A feature is **monosemantic** if it correlates strongly with exactly one concept. This is the key test: do SAE features cleanly separate algorithmic concepts?

In [ ]:
# Categorize features (using mono, dead, concept_matrix from earlier cells)
poly_feats = [i for i in range(dict_size) 
              if i not in mono and i not in dead 
              and max_corr_per_feat[i] > 0.1]
other_feats = [i for i in range(dict_size) 
               if i not in mono and i not in dead and i not in poly_feats]

print(f"Feature breakdown ({dict_size} total):")
print(f"  Monosemantic (|corr| > 0.5 with exactly 1 concept): {len(mono)}")
print(f"  Polysemantic (|corr| > 0.1 with multiple concepts):  {len(poly_feats)}")
print(f"  Weak/uncorrelated:                                    {len(other_feats)}")
print(f"  Dead (never activate):                                {len(dead)}")

# Detailed look at monosemantic features
if mono:
    print(f"\nMonosemantic features detail:")
    for feat_idx in mono[:15]:
        corrs = {name: concept_matrix[feat_idx, j].item()
                 for j, name in enumerate(concept_names)}
        best_concept = max(corrs, key=lambda k: abs(corrs[k]))
        print(f"  Feature {feat_idx:4d} -> {best_concept} (r={corrs[best_concept]:+.3f})")


### 3.6 Max correlation distribution

Distribution of the maximum absolute correlation each feature achieves with any concept. High values = features that cleanly track algorithmic state.

In [ ]:
# Filter out dead features for the distribution
active_max_corrs = max_corr_per_feat[[i for i in range(dict_size) if i not in dead]]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(active_max_corrs, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(x=0.5, color='r', linestyle='--', alpha=0.7, label="Monosemantic threshold (0.5)")
ax.axvline(x=0.3, color='orange', linestyle='--', alpha=0.7, label="Weak threshold (0.3)")
ax.set_xlabel("Max |correlation| with any concept")
ax.set_ylabel("Number of features")
ax.set_title("Feature-Concept Correlation Strength (active features)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
for j, name in enumerate(concept_names):
    top_idx = concept_matrix[:, j].abs().topk(3).indices
    print(f"\n{name}:")
    for idx in top_idx:
        row = {n: concept_matrix[idx, k].item() for k, n in enumerate(concept_names)}
        print(f"  Feature {idx.item():4d}: {row}")


---

## 4. SAE Hyperparameter Sweep

Compare different SAE architectures and hyperparameters using the same NAR activations.
We sweep over:
- **Expansion factors**: 1x, 4x, 16x (dict_size = hidden_dim * factor)
- **k values**: 8, 16, 32, 64
- **SAE types**: BatchTopK vs Standard (L1-penalized)

In [ ]:
# --- Sweep configuration ---
SWEEP_STEPS = SAE_STEPS  # same training budget for fair comparison

sweep_configs = [
    # BatchTopK: vary expansion factor and k
    {"name": "btk_1x_k8",  "type": "batchtopk", "expansion": 1,  "k": 8},
    {"name": "btk_1x_k16", "type": "batchtopk", "expansion": 1,  "k": 16},
    {"name": "btk_4x_k8",  "type": "batchtopk", "expansion": 4,  "k": 8},
    {"name": "btk_4x_k16", "type": "batchtopk", "expansion": 4,  "k": 16},
    {"name": "btk_4x_k32", "type": "batchtopk", "expansion": 4,  "k": 32},
    {"name": "btk_4x_k64", "type": "batchtopk", "expansion": 4,  "k": 64},
    {"name": "btk_16x_k16","type": "batchtopk", "expansion": 16, "k": 16},
    {"name": "btk_16x_k32","type": "batchtopk", "expansion": 16, "k": 32},
    {"name": "btk_16x_k64","type": "batchtopk", "expansion": 16, "k": 64},
    # Standard SAE baselines (L1-penalized)
    {"name": "std_1x",  "type": "standard", "expansion": 1,  "sparsity_coeff": 1e-3},
    {"name": "std_4x",  "type": "standard", "expansion": 4,  "sparsity_coeff": 1e-3},
    {"name": "std_16x", "type": "standard", "expansion": 16, "sparsity_coeff": 1e-3},
]

print(f"Sweep: {len(sweep_configs)} configurations, {SWEEP_STEPS} steps each")
for cfg in sweep_configs:
    ds = HIDDEN_DIM * cfg["expansion"]
    print(f"  {cfg['name']:15s} | {cfg['type']:10s} | dict_size={ds:5d} | {'k='+str(cfg.get('k','')) if 'k' in cfg else 'L1='+str(cfg['sparsity_coeff'])}")

### 4.1 Run sweep

In [ ]:
# Load concept labels once (reuse across all configs)
concept_data = torch.load(RESULTS_DIR / "concept_labels.pt", map_location="cpu", weights_only=True)
concept_labels_sweep = concept_data["labels"]
concept_labels_sweep.pop("is_unvisited", None)
concept_names_sweep = list(concept_labels_sweep.keys())

N_CORR = 10_000
min_n = min(N_CORR, activations.shape[0], min(v.shape[0] for v in concept_labels_sweep.values()))
labels_sweep = torch.stack([concept_labels_sweep[name][:min_n].float() for name in concept_names_sweep], dim=1)

sweep_results = []
sweep_dir = RESULTS_DIR.parent.parent / "sweep"
sweep_dir.mkdir(parents=True, exist_ok=True)

for cfg in sweep_configs:
    name = cfg["name"]
    ds = HIDDEN_DIM * cfg["expansion"]
    print(f"\n{'='*60}")
    print(f"Training: {name} (dict_size={ds})")
    print(f"{'='*60}")

    # Create SAE
    if cfg["type"] == "batchtopk":
        sae_sweep = BatchTopKSAE(input_dim=HIDDEN_DIM, dict_size=ds, k=cfg["k"]).to(device)
    else:
        sae_sweep = SparseAutoencoder(input_dim=HIDDEN_DIM, dict_size=ds, sparsity_coeff=cfg["sparsity_coeff"]).to(device)

    n_params = sum(p.numel() for p in sae_sweep.parameters())
    print(f"  Parameters: {n_params:,}")

    # Train
    trainer_sweep = SAETrainer(sae_sweep, lr=SAE_LR, resample_dead_every=0)
    sae_loader_sweep = make_activation_dataloader(activations, batch_size=SAE_BATCH_SIZE)

    step = 0
    while step < SWEEP_STEPS:
        for (batch,) in sae_loader_sweep:
            if step >= SWEEP_STEPS:
                break
            batch = batch.to(device)
            output = trainer_sweep.train_step(batch)
            if step % 5_000 == 0:
                print(f"  [{name}] step {step}/{SWEEP_STEPS} | loss={output.loss.item():.4f} | L0={output.l0.item():.1f}")
            step += 1

    final_stats = trainer_sweep.get_training_stats()
    print(f"  Final loss: {final_stats.get('recent_loss', 0):.4f}, L0: {final_stats.get('recent_l0', 0):.1f}")

    # Evaluate reconstruction
    sae_sweep.eval()
    torch.cuda.empty_cache()
    recon_list, feat_list = [], []
    with torch.no_grad():
        for i in range(0, min_n, SAE_BATCH_SIZE):
            batch = activations[i:i+SAE_BATCH_SIZE].to(device)
            out = sae_sweep(batch)
            recon_list.append(out.reconstructed.cpu())
            feat_list.append(out.features.cpu())
            del out, batch
    reconstructed = torch.cat(recon_list)[:min_n]
    features_sweep = torch.cat(feat_list)[:min_n].float()
    del recon_list, feat_list

    sample = activations[:min_n]
    recon_mse = F.mse_loss(reconstructed, sample).item()
    total_var = sample.var(dim=0).sum()
    residual_var = (sample - reconstructed).var(dim=0).sum()
    fve = (1 - residual_var / total_var).item()
    del reconstructed

    num_dead = (features_sweep.sum(dim=0) == 0).sum().item()
    l0_per = (features_sweep > 0).float().sum(dim=-1)
    l0_mean = l0_per.mean().item()

    # Concept correlations
    feat_c = features_sweep - features_sweep.mean(dim=0, keepdim=True)
    lab_c = labels_sweep - labels_sweep.mean(dim=0, keepdim=True)
    feat_n = feat_c / feat_c.std(dim=0, keepdim=True).clamp(min=1e-8)
    lab_n = lab_c / lab_c.std(dim=0, keepdim=True).clamp(min=1e-8)
    cm = (feat_n.T @ lab_n) / min_n
    del feat_c, lab_c, feat_n, lab_n

    # Monosemantic count (relaxed criterion)
    n_mono = 0
    for i in range(ds):
        corrs = cm[i].abs()
        if corrs.max() > 0.5 and (corrs > 0.3).sum() <= 2:
            n_mono += 1

    # Max correlation per concept
    max_corrs = {concept_names_sweep[j]: cm[:, j].abs().max().item() for j in range(len(concept_names_sweep))}

    result = {
        "name": name,
        "type": cfg["type"],
        "expansion": cfg["expansion"],
        "k": cfg.get("k", None),
        "sparsity_coeff": cfg.get("sparsity_coeff", None),
        "dict_size": ds,
        "params": n_params,
        "mse": recon_mse,
        "fve": fve,
        "dead_pct": 100 * num_dead / ds,
        "l0": l0_mean,
        "n_mono": n_mono,
        "max_corrs": max_corrs,
        "final_loss": final_stats.get("recent_loss", 0),
    }
    sweep_results.append(result)
    del features_sweep, sae_sweep, cm
    torch.cuda.empty_cache()

    print(f"  FVE: {fve:.4f} | Dead: {num_dead}/{ds} ({100*num_dead/ds:.1f}%) | L0: {l0_mean:.1f} | Mono: {n_mono}")
    print(f"  Max |corr|: {max_corrs}")

    # Save checkpoint
    torch.save(result, sweep_dir / f"{name}_result.pt")

# Save all sweep results
torch.save(sweep_results, sweep_dir / "all_results.pt")
print(f"\nSaved all sweep results to {sweep_dir / 'all_results.pt'}")

### 4.2 Sweep results summary

In [ ]:
# --- Summary table ---
# Load from disk if needed: sweep_results = torch.load(sweep_dir / "all_results.pt")

print(f"{'Config':15s} | {'Type':10s} | {'DictSz':>6s} | {'FVE':>6s} | {'Dead%':>5s} | {'L0':>5s} | {'Mono':>4s} | {'MaxCorr':>7s}")
print("-" * 85)
for r in sweep_results:
    best_corr = max(r["max_corrs"].values())
    print(f"{r['name']:15s} | {r['type']:10s} | {r['dict_size']:6d} | {r['fve']:.4f} | {r['dead_pct']:5.1f} | {r['l0']:5.1f} | {r['n_mono']:4d} | {best_corr:.4f}")

### 4.3 Sweep visualizations

In [ ]:
# --- Sweep plots ---
btk = [r for r in sweep_results if r["type"] == "batchtopk"]
std = [r for r in sweep_results if r["type"] == "standard"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. FVE vs dict_size
ax = axes[0, 0]
for exp in [1, 4, 16]:
    pts = [r for r in btk if r["expansion"] == exp]
    if pts:
        ax.scatter([r["k"] for r in pts], [r["fve"] for r in pts], label=f"BTK {exp}x", s=60)
for r in std:
    ax.scatter(r["l0"], r["fve"], marker="x", s=80, c="black", zorder=5)
ax.set_xlabel("k (or L0 for standard)")
ax.set_ylabel("FVE")
ax.set_title("Reconstruction Quality")
ax.legend()

# 2. Dead features % vs dict_size
ax = axes[0, 1]
for exp in [1, 4, 16]:
    pts = [r for r in btk if r["expansion"] == exp]
    if pts:
        ax.scatter([r["k"] for r in pts], [r["dead_pct"] for r in pts], label=f"BTK {exp}x", s=60)
for r in std:
    ax.scatter(r["l0"], r["dead_pct"], marker="x", s=80, c="black", zorder=5)
ax.set_xlabel("k (or L0 for standard)")
ax.set_ylabel("Dead features (%)")
ax.set_title("Feature Utilization")
ax.legend()

# 3. Monosemantic features vs config
ax = axes[0, 2]
names = [r["name"] for r in sweep_results]
monos = [r["n_mono"] for r in sweep_results]
colors = ["tab:blue" if r["type"] == "batchtopk" else "tab:orange" for r in sweep_results]
ax.barh(range(len(names)), monos, color=colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel("Monosemantic features")
ax.set_title("Monosemanticity")

# 4. Max correlation per concept
ax = axes[1, 0]
for j, concept in enumerate(concept_names_sweep):
    vals = [r["max_corrs"][concept] for r in sweep_results]
    ax.bar([i + j*0.25 for i in range(len(sweep_results))], vals, width=0.25, label=concept)
ax.set_xticks(range(len(sweep_results)))
ax.set_xticklabels([r["name"] for r in sweep_results], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Max |correlation|")
ax.set_title("Concept Alignment")
ax.legend(fontsize=8)

# 5. FVE vs monosemantic features (Pareto front)
ax = axes[1, 1]
for r in btk:
    ax.scatter(r["fve"], r["n_mono"], s=60, c="tab:blue")
    ax.annotate(r["name"], (r["fve"], r["n_mono"]), fontsize=7, ha="left")
for r in std:
    ax.scatter(r["fve"], r["n_mono"], s=60, c="tab:orange", marker="x")
    ax.annotate(r["name"], (r["fve"], r["n_mono"]), fontsize=7, ha="left")
ax.set_xlabel("FVE (reconstruction)")
ax.set_ylabel("Monosemantic features")
ax.set_title("Reconstruction vs Interpretability")

# 6. Dead % vs monosemantic
ax = axes[1, 2]
for r in btk:
    ax.scatter(r["dead_pct"], r["n_mono"], s=60, c="tab:blue")
    ax.annotate(r["name"], (r["dead_pct"], r["n_mono"]), fontsize=7, ha="left")
for r in std:
    ax.scatter(r["dead_pct"], r["n_mono"], s=60, c="tab:orange", marker="x")
    ax.annotate(r["name"], (r["dead_pct"], r["n_mono"]), fontsize=7, ha="left")
ax.set_xlabel("Dead features (%)")
ax.set_ylabel("Monosemantic features")
ax.set_title("Feature Utilization vs Interpretability")

plt.tight_layout()
plt.savefig(sweep_dir / "sweep_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved plots to {sweep_dir / 'sweep_plots.png'}")

In [ ]:
BEST_NAME = "btk_4x_k32"
BEST_EXP  = 4
BEST_K    = 32
best_ds   = HIDDEN_DIM * BEST_EXP
_best_ckpt = sweep_dir / f"{BEST_NAME}_weights.pt"

sae_best = BatchTopKSAE(input_dim=HIDDEN_DIM, dict_size=best_ds, k=BEST_K).to(device)

if _best_ckpt.exists():
    _c = torch.load(_best_ckpt, map_location=device, weights_only=True)
    sae_best.load_state_dict(_c["state_dict"])
    sae_best.eval()
    print(f"Loaded {BEST_NAME} weights from {_best_ckpt.name}")
else:
    trainer_best   = SAETrainer(sae_best, lr=SAE_LR, resample_dead_every=0)
    sae_loader_best = make_activation_dataloader(activations, batch_size=SAE_BATCH_SIZE)
    step = 0
    while step < SWEEP_STEPS:
        for (batch,) in sae_loader_best:
            if step >= SWEEP_STEPS:
                break
            output = trainer_best.train_step(batch.to(device))
            if step % 5_000 == 0:
                print(f"Step {step}/{SWEEP_STEPS} | loss={output.loss.item():.4f} | L0={output.l0.item():.1f}")
            step += 1
    torch.save({"state_dict": sae_best.state_dict(), "config": sae_best.get_config()},
               _best_ckpt)
    print(f"Saved {BEST_NAME} weights")

In [ ]:
# Encode all samples with best SAE
sae_best.eval()
torch.cuda.empty_cache()

concept_data = torch.load(RESULTS_DIR / "concept_labels.pt", map_location="cpu", weights_only=True)
concept_labels_best = concept_data["labels"]
concept_labels_best.pop("is_unvisited", None)
concept_names_best = list(concept_labels_best.keys())

N = 10_000
min_n_best = min(N, activations.shape[0], min(v.shape[0] for v in concept_labels_best.values()))
labels_best = torch.stack([concept_labels_best[name][:min_n_best].float() for name in concept_names_best], dim=1)

feat_list = []
with torch.no_grad():
    for i in range(0, min_n_best, SAE_BATCH_SIZE):
        batch = activations[i:i+SAE_BATCH_SIZE].to(device)
        out = sae_best(batch)
        feat_list.append(out.features.cpu())
        del out, batch
features_best = torch.cat(feat_list).float()[:min_n_best]
del feat_list
torch.cuda.empty_cache()

# Full correlation matrix (best_ds x num_concepts)
feat_c = features_best - features_best.mean(0, keepdim=True)
lab_c = labels_best - labels_best.mean(0, keepdim=True)
feat_n = feat_c / feat_c.std(0, keepdim=True).clamp(min=1e-8)
lab_n = lab_c / lab_c.std(0, keepdim=True).clamp(min=1e-8)
cm_best = (feat_n.T @ lab_n) / min_n_best
del feat_c, lab_c, feat_n, lab_n

print(f"Correlation matrix: {cm_best.shape}")


In [ ]:
# For each concept, show top 10 features with full correlation profiles
for j, concept in enumerate(concept_names_best):
    top_idx = cm_best[:, j].abs().topk(10).indices
    print(f"\n{'='*60}")
    print(f"Concept: {concept}")
    print(f"{'Feat':>5s} | {'corr_'+concept:>12s} | " + " | ".join(f"{'corr_'+c:>12s}" for c in concept_names_best if c != concept))
    print("-" * 60)
    for idx in top_idx:
        row = cm_best[idx]
        main = row[j].item()
        others = [f"{row[k].item():+.4f}" for k in range(len(concept_names_best)) if k != j]
        print(f"  {idx.item():4d} | {main:+.4f}       | " + " | ".join(f"{o:>12s}" for o in others))


In [ ]:
# Show activation distribution of top feature vs concept membership
top_per_concept = {}
for j, concept in enumerate(concept_names_best):
    top_idx = cm_best[:, j].abs().argmax().item()
    top_per_concept[concept] = top_idx

fig, axes = plt.subplots(1, len(concept_names_best), figsize=(5*len(concept_names_best), 4))
for ax, (concept, feat_idx) in zip(axes, top_per_concept.items()):
    feat_acts = features_best[:, feat_idx].numpy()
    labels_c = labels_best[:, concept_names_best.index(concept)].numpy().astype(bool)
    
    ax.hist(feat_acts[~labels_c], bins=50, alpha=0.6, label=f"concept=0", density=True)
    ax.hist(feat_acts[labels_c],  bins=50, alpha=0.6, label=f"concept=1", density=True)
    corr = cm_best[feat_idx, concept_names_best.index(concept)].item()
    ax.set_title(f"Feature {feat_idx}\n→ {concept}\n(r={corr:+.3f})")
    ax.set_xlabel("Activation value")
    ax.legend()

plt.suptitle("btk_4x_k32: Top feature per concept — activation distributions", y=1.02)
plt.tight_layout()
plt.savefig(sweep_dir / "best_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Save btk_4x_k32 detailed results
torch.save({
    "concept_matrix": cm_best,
    "concept_names": concept_names_best,
    "features": features_best,
}, sweep_dir / "btk_4x_k32_detailed.pt")
print(f"Saved detailed btk_4x_k32 results to {sweep_dir / 'btk_4x_k32_detailed.pt'}")

---

## 5. Per-Layer Activation Analysis

Each BFS concept may be most strongly encoded at a specific processor layer.
We collect the `node_mlp_output` from each of the `NUM_LAYERS` layers separately
and correlate each with BFS concept labels.

### 5.1 Collect per-layer activations

In [ ]:
_pla_path = RESULTS_DIR / "per_layer_activations.pt"
if _pla_path.exists() and ('per_layer_acts' not in dir() or not isinstance(per_layer_acts, dict)):
    per_layer_acts = torch.load(_pla_path)
    for i in range(NUM_LAYERS):
        print(f"Layer {i}: {per_layer_acts[i].shape}")
    print(f"Loaded from {_pla_path}")
else:
    # Collect node_mlp_output from each processor layer across all steps
    model.eval()
    layer_act_lists = {i: [] for i in range(NUM_LAYERS)}
    
    with torch.no_grad():
        for batch in act_loader:
            inputs, _, _ = batch_to_model_inputs(batch, spec, device)
            num_steps = batch.lengths.max().item()
    
            output = model(
                inputs=inputs,
                output_types=output_types,
                num_steps=num_steps,
                return_activations=True,
            )
            acts = output.activations
    
            for layer_idx in range(NUM_LAYERS):
                for step_dict in acts['layer_activations'][layer_idx]:
                    h = step_dict['node_mlp_output']  # (batch, nodes, hidden_dim)
                    layer_act_lists[layer_idx].append(h.reshape(-1, h.shape[-1]).cpu())
    
    per_layer_acts = {i: torch.cat(layer_act_lists[i], dim=0) for i in range(NUM_LAYERS)}
    for i in range(NUM_LAYERS):
        print(f"Layer {i}: {per_layer_acts[i].shape}")
    
    torch.save(per_layer_acts, RESULTS_DIR / "per_layer_activations.pt")
    print(f"\nSaved to {RESULTS_DIR / 'per_layer_activations.pt'}")


### 5.2 Per-layer concept correlations

In [ ]:
_plc_path = sweep_dir / "per_layer_correlations.pt"
if _plc_path.exists() and 'layer_corr_results' not in dir():
    layer_corr_results = torch.load(_plc_path, weights_only=False)
    concept_names_layer = layer_corr_results[0]["concept_names"] if "concept_names" in layer_corr_results[0] else None
    if concept_names_layer is None:
        concept_data = torch.load(RESULTS_DIR / "concept_labels.pt", map_location="cpu", weights_only=True)
        _tmp = {k: v for k, v in concept_data["labels"].items() if k != "is_unvisited"}
        concept_names_layer = list(_tmp.keys())
    print(f"Loaded per-layer correlations from disk")
    for li, res in layer_corr_results.items():
        print(f"  L{li}: " + "  ".join(f"{c}={res['max_corrs'][j]:.3f}" for j, c in enumerate(concept_names_layer)))
else:
    # Correlate each layer's activations with BFS concept labels
    # Reuse labels from section 3 (labels_sweep / labels_best — same alignment)
    concept_data = torch.load(RESULTS_DIR / "concept_labels.pt", map_location="cpu", weights_only=True)
    concept_labels_layer = concept_data["labels"]
    concept_labels_layer.pop("is_unvisited", None)
    concept_names_layer = list(concept_labels_layer.keys())
    
    N = 10_000
    layer_corr_results = {}
    
    print(f"{'Layer':>5s} | " + " | ".join(f"{c:>14s}" for c in concept_names_layer))
    print("-" * (8 + 17 * len(concept_names_layer)))
    
    for layer_idx in range(NUM_LAYERS):
        h = per_layer_acts[layer_idx]
        n = min(N, h.shape[0], min(v.shape[0] for v in concept_labels_layer.values()))
        h_sub = h[:n].float()
        l_sub = torch.stack([concept_labels_layer[c][:n].float() for c in concept_names_layer], dim=1)
    
        h_c = h_sub - h_sub.mean(0, keepdim=True)
        l_c = l_sub - l_sub.mean(0, keepdim=True)
        h_n = h_c / h_c.std(0, keepdim=True).clamp(min=1e-8)
        l_n = l_c / l_c.std(0, keepdim=True).clamp(min=1e-8)
        cm_layer = (h_n.T @ l_n) / n
    
        max_corrs = [cm_layer[:, j].abs().max().item() for j in range(len(concept_names_layer))]
        layer_corr_results[layer_idx] = {"cm": cm_layer, "max_corrs": max_corrs}
        print(f"  L{layer_idx}   | " + " | ".join(f"{c:>+14.4f}" for c in max_corrs))
    
    torch.save(layer_corr_results, sweep_dir / "per_layer_correlations.pt")
    print(f"\nSaved to {sweep_dir / 'per_layer_correlations.pt'}")


### 5.3 Per-layer correlation heatmap

In [ ]:
if 'layer_corr_results' not in dir():
    layer_corr_results = torch.load(sweep_dir / "per_layer_correlations.pt", weights_only=False)
if 'concept_names_layer' not in dir():
    _cd = torch.load(RESULTS_DIR / "concept_labels.pt", map_location="cpu", weights_only=True)
    concept_names_layer = [k for k in _cd["labels"] if k != "is_unvisited"]

# Heatmap: layers × concepts, showing max |correlation|
import numpy as np

matrix = np.array([[layer_corr_results[i]["max_corrs"][j]
                    for j in range(len(concept_names_layer))]
                   for i in range(NUM_LAYERS)])

fig, ax = plt.subplots(figsize=(len(concept_names_layer) * 2 + 1, NUM_LAYERS + 1))
im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(concept_names_layer)))
ax.set_xticklabels(concept_names_layer, rotation=30, ha='right')
ax.set_yticks(range(NUM_LAYERS))
ax.set_yticklabels([f"Layer {i}" for i in range(NUM_LAYERS)])
ax.set_title("Max |Pearson r| between layer activations and BFS concepts")

for i in range(NUM_LAYERS):
    for j in range(len(concept_names_layer)):
        ax.text(j, i, f"{matrix[i, j]:.3f}", ha='center', va='center', fontsize=10)

plt.colorbar(im, ax=ax, label="Max |correlation|")
plt.tight_layout()
plt.savefig(sweep_dir / "per_layer_concept_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Summary & Next Steps

**Key metrics to check:**
- NAR validation accuracy > 0.9 (BFS should be near-perfect)
- SAE fraction of variance explained > 0.9
- Dead features < 30%
- Monosemantic features > 0 (ideally 10+)

**Findings so far:**
- Feature 49 (btk_4x_k32, final output) cleanly encodes `is_source` (r=+0.73)
- Layer 0 has the strongest linear signal for `is_visited` (r=0.47) and `is_frontier` (r=0.26)
- Later layers compress visited/frontier into abstract task-relevant representations
- SAE on Layer 0 expected to find monosemantic features for `is_visited`/`is_frontier`
- Standard SAE fails entirely (L0 >> k, 0 monosemantic)
- btk_4x_k32 is the best config (FVE=0.98, 14.8% dead)

**Next experiments:**
- Repeat for Dijkstra, DFS, MST (change `ALGORITHM` at top)
- Multi-algorithm training (shared processor across BFS + Dijkstra + DFS)
- Per-step temporal analysis: do features track BFS wavefront propagation over time?

---

## 6. SAE on Each Processor Layer

Train a BatchTopK SAE (4×, k=32 — best config from Section 4) on the
`node_mlp_output` of **every** processor layer.  Results are stored in
`layer_saes[layer_idx]` and `layer_eval[layer_idx]` so the rest of the
notebook works for any `NUM_LAYERS` without modification.

Checkpoints are saved to `sae_layer{i}/` under the run root.  If a
checkpoint already exists the cell skips retraining and loads instead.

### 6.1 Train SAE on every layer

In [ ]:
if 'per_layer_acts' not in dir() or not isinstance(per_layer_acts, dict):
    per_layer_acts = torch.load(RESULTS_DIR / "per_layer_activations.pt")

LAYER_EXPANSION = 4
LAYER_K         = 16 if LOCAL_DEBUG else 32
layer_dict_size = HIDDEN_DIM * LAYER_EXPANSION

layer_saes     = {}   # layer_idx -> BatchTopKSAE (trained, eval mode)
layer_sae_dirs = {}   # layer_idx -> Path

for layer_idx in range(NUM_LAYERS):
    sae_dir   = RESULTS_DIR.parent.parent / f"sae_layer{layer_idx}"
    ckpt_path = sae_dir / f"sae_l{layer_idx}_final.pt"
    sae_dir.mkdir(parents=True, exist_ok=True)
    layer_sae_dirs[layer_idx] = sae_dir

    sae = BatchTopKSAE(
        input_dim=HIDDEN_DIM, dict_size=layer_dict_size, k=LAYER_K
    ).to(device)

    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
        sae.load_state_dict(ckpt["state_dict"])
        sae.eval()
        layer_saes[layer_idx] = sae
        print(f"L{layer_idx}: loaded checkpoint {ckpt_path.name}")
        continue

    print(f"\n{'─'*52}\nTraining SAE for Layer {layer_idx}\n{'─'*52}")
    acts    = per_layer_acts[layer_idx]
    trainer = SAETrainer(sae, lr=SAE_LR, resample_dead_every=0)
    loader  = make_activation_dataloader(acts, batch_size=SAE_BATCH_SIZE)

    step = 0
    while step < SAE_STEPS:
        for (batch,) in loader:
            if step >= SAE_STEPS:
                break
            out = trainer.train_step(batch.to(device))
            if step % 5_000 == 0:
                print(f"  step {step:>6d}/{SAE_STEPS} | "
                      f"loss={out.loss.item():.4f} | L0={out.l0.item():.1f}")
            step += 1

    torch.save({"state_dict": sae.state_dict(), "config": sae.get_config()}, ckpt_path)
    sae.eval()
    layer_saes[layer_idx] = sae
    print(f"  Saved → {ckpt_path}")

# ── Backward-compat aliases (used by Sections 8, 10, 11) ────────────────────
sae_l0          = layer_saes[0]
l0_dict_size    = layer_dict_size
l0_sae_dir      = layer_sae_dirs[0]
print(f"\nAll SAEs ready: layers {sorted(layer_saes.keys())}")

### 6.2 Evaluate all layer SAEs

In [ ]:
# Load layer_eval from disk if available and not in memory
if 'layer_eval' not in dir():
    _le_loaded = True
    layer_eval = {}
    for _li in range(NUM_LAYERS):
        _p = layer_sae_dirs[_li] / f"l{_li}_correlation_results.pt"
        if _p.exists():
            layer_eval[_li] = torch.load(_p, weights_only=False)
        else:
            _le_loaded = False
            break
    if _le_loaded and len(layer_eval) == NUM_LAYERS:
        concept_names    = layer_eval[0]["concept_names"]
        cm_l0            = layer_eval[0]["cm"]
        concept_names_l0 = concept_names
        mono_l0          = layer_eval[0]["mono"]
        dead_l0          = layer_eval[0]["dead"]
        for _li in range(NUM_LAYERS):
            _e = layer_eval[_li]
            print(f"L{_li}: FVE={_e['fve']:.4f}  dead={_e['n_dead']}/{_e['n_dead']+len([x for x in range(HIDDEN_DIM*LAYER_EXPANSION) if x not in _e['dead']]) }  mono={len(_e['mono'])}")
        print("Loaded layer_eval from disk")
    else:
        layer_eval = {}  # reset; will be recomputed below

if not layer_eval:
    N_EVAL = 10_000
    concept_data = torch.load(
        RESULTS_DIR / "concept_labels.pt", map_location="cpu", weights_only=True
    )
    _clabels = {k: v for k, v in concept_data["labels"].items() if k != "is_unvisited"}
    concept_names = list(_clabels.keys())
    n_labels = min(N_EVAL, min(v.shape[0] for v in _clabels.values()))
    labels_t = torch.stack([_clabels[c][:n_labels].float() for c in concept_names], dim=1)
    
    MONO_THRESH = 0.4
    layer_eval = {}   # layer_idx -> dict of metrics
    
    for layer_idx in range(NUM_LAYERS):
        sae  = layer_saes[layer_idx]
        acts = per_layer_acts[layer_idx]
        samp = acts[:N_EVAL]
    
        recon_list, feat_list = [], []
        with torch.no_grad():
            for i in range(0, N_EVAL, SAE_BATCH_SIZE):
                b = samp[i:i + SAE_BATCH_SIZE].to(device)
                o = sae(b)
                recon_list.append(o.reconstructed.cpu())
                feat_list.append(o.features.cpu())
                del o, b
        recon = torch.cat(recon_list)
        feats = torch.cat(feat_list).float()
        del recon_list, feat_list
        torch.cuda.empty_cache()
    
        mse  = F.mse_loss(recon, samp).item()
        fve  = (1 - (samp - recon).var(dim=0).sum() / samp.var(dim=0).sum()).item()
        n_dead = (feats.sum(0) == 0).sum().item()
        spars  = (feats > 0).float().sum(-1).mean().item()
    
        # Pearson correlation matrix: (dict_size, n_concepts)
        n  = min(N_EVAL, n_labels)
        f  = feats[:n];  l = labels_t[:n]
        fc = f - f.mean(0, keepdim=True);  lc = l - l.mean(0, keepdim=True)
        fn = fc / fc.std(0, keepdim=True).clamp(min=1e-8)
        ln = lc / lc.std(0, keepdim=True).clamp(min=1e-8)
        cm = (fn.T @ ln) / n
        del fc, lc, fn, ln
    
        mono = [i for i in range(layer_dict_size)
                if cm[i].abs().max() > MONO_THRESH and (cm[i].abs() > 0.3).sum() <= 2]
        dead = (feats.sum(0) == 0).nonzero(as_tuple=True)[0].tolist()
    
        layer_eval[layer_idx] = {
            "fve": fve, "mse": mse, "n_dead": n_dead,
            "dead_pct": 100 * n_dead / layer_dict_size,
            "sparsity": spars, "cm": cm, "mono": mono, "dead": dead,
            "features": feats, "concept_names": concept_names,
        }
    
        # Save to disk
        torch.save(
            {k: v for k, v in layer_eval[layer_idx].items()},
            layer_sae_dirs[layer_idx] / f"l{layer_idx}_correlation_results.pt",
        )
    
        print(f"L{layer_idx}: FVE={fve:.4f}  dead={n_dead}/{layer_dict_size} "
              f"({100*n_dead/layer_dict_size:.1f}%)  mono={len(mono)}")
    
    # ── More backward-compat aliases ────────────────────────────────────────────
    cm_l0           = layer_eval[0]["cm"]
    concept_names_l0 = concept_names
    mono_l0         = layer_eval[0]["mono"]
    dead_l0         = layer_eval[0]["dead"]


### 6.3 Concept correlations — per-layer summary

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Text table ────────────────────────────────────────────────────────────────
col_w = 10
header = f"{'Concept':14s}" + "".join(f"{'L'+str(i):>{col_w}}" for i in range(NUM_LAYERS))
sep    = "─" * len(header)
print("Max |Pearson r| per (layer, concept)  [MONO_THRESH=0.4]")
print(sep); print(header); print(sep)
for j, concept in enumerate(concept_names):
    row = f"{concept:14s}"
    for li in range(NUM_LAYERS):
        max_r = layer_eval[li]["cm"][:, j].abs().max().item()
        row += f"{max_r:>{col_w}.3f}"
    print(row)
print(sep)
print(f"{'Mono count':14s}" +
      "".join(f"{len(layer_eval[li]['mono']):>{col_w}}" for li in range(NUM_LAYERS)))
print(f"{'Dead %':14s}" +
      "".join(f"{layer_eval[li]['dead_pct']:>{col_w}.1f}" for li in range(NUM_LAYERS)))

# ── Monosemantic feature details per layer ────────────────────────────────────
for li in range(NUM_LAYERS):
    mono = layer_eval[li]["mono"]
    if not mono:
        continue
    print(f"\nL{li} monosemantic features (|r|>{MONO_THRESH}):")
    for fi in mono[:10]:
        corrs = {c: layer_eval[li]["cm"][fi, j].item() for j, c in enumerate(concept_names)}
        best  = max(corrs, key=lambda k: abs(corrs[k]))
        print(f"  Feature {fi:4d} → {best} (r={corrs[best]:+.3f})")

# ── Heatmap: (concepts × layers) of max |r| ──────────────────────────────────
mat = np.array([
    [layer_eval[li]["cm"][:, j].abs().max().item()
     for li in range(NUM_LAYERS)]
    for j in range(len(concept_names))
])
fig, ax = plt.subplots(figsize=(max(5, NUM_LAYERS * 1.8), max(3, len(concept_names) * 0.9)))
im = ax.imshow(mat, vmin=0, vmax=1.0, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(NUM_LAYERS))
ax.set_xticklabels([f"L{i}" for i in range(NUM_LAYERS)])
ax.set_yticks(range(len(concept_names)))
ax.set_yticklabels(concept_names)
ax.set_title(f"Max |Pearson r| (SAE features, 4×/k={LAYER_K})")
plt.colorbar(im, ax=ax, fraction=0.03)
for i in range(len(concept_names)):
    for j in range(NUM_LAYERS):
        ax.text(j, i, f"{mat[i,j]:.2f}", ha="center", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS_DIR.parent.parent / "sae_layer_concept_heatmap.png",
            dpi=150, bbox_inches="tight")
plt.show()

---

## 7. Smaller SAE on Layer 0 (1× expansion, k=8)

The 4× SAE had 85.7% dead features, meaning it is over-parameterised for Layer 0.
We try a 1× expansion (dict_size = HIDDEN_DIM = 128) with k=8 to reduce redundancy
and force the SAE to use features more efficiently.

### 7.1 Setup and train

In [ ]:
# Load Layer 0 activations (reuse from Section 5.1 if already in memory)
if 'per_layer_acts' not in dir() or not isinstance(per_layer_acts, dict):
    per_layer_acts = torch.load(RESULTS_DIR / "per_layer_activations.pt")
l0_acts = per_layer_acts[0]
print(f"Layer 0 activations: {l0_acts.shape}")

L0_SMALL_EXPANSION = 1
L0_SMALL_K = 4 if LOCAL_DEBUG else 8
l0_small_dict_size = HIDDEN_DIM * L0_SMALL_EXPANSION
l0_small_sae_dir = RESULTS_DIR.parent.parent / "sae_layer0_small"
l0_small_sae_dir.mkdir(parents=True, exist_ok=True)

sae_l0_small = BatchTopKSAE(
    input_dim=HIDDEN_DIM, dict_size=l0_small_dict_size, k=L0_SMALL_K
).to(device)
print(f"SAE (small): input_dim={HIDDEN_DIM}, dict_size={l0_small_dict_size}, k={L0_SMALL_K}")
print(f"Parameters: {sum(p.numel() for p in sae_l0_small.parameters()):,}")

In [ ]:
_small_ckpt = l0_small_sae_dir / "sae_l0_small_final.pt"
if _small_ckpt.exists():
    _c = torch.load(_small_ckpt, map_location=device, weights_only=True)
    sae_l0_small.load_state_dict(_c["state_dict"])
    sae_l0_small.eval()
    print(f"Loaded small L0 SAE from {_small_ckpt.name}")
else:
    trainer_l0_small = SAETrainer(sae_l0_small, lr=SAE_LR, resample_dead_every=0)
    l0_small_loader  = make_activation_dataloader(l0_acts, batch_size=SAE_BATCH_SIZE)
    step = 0
    while step < SAE_STEPS:
        for (batch,) in l0_small_loader:
            if step >= SAE_STEPS:
                break
            output = trainer_l0_small.train_step(batch.to(device))
            if step % 5_000 == 0:
                print(f"Step {step}/{SAE_STEPS} | loss={output.loss.item():.4f} | L0={output.l0.item():.1f}")
            step += 1
    torch.save({"state_dict": sae_l0_small.state_dict(), "config": sae_l0_small.get_config()},
               _small_ckpt)
    print(f"Saved to {_small_ckpt}")

### 7.2 Reconstruction quality

In [ ]:
if 'sae_l0_small' not in dir():
    _c = torch.load(l0_small_sae_dir / "sae_l0_small_final.pt", map_location=device, weights_only=True)
    sae_l0_small = BatchTopKSAE(input_dim=HIDDEN_DIM, dict_size=l0_small_dict_size, k=L0_SMALL_K).to(device)
    sae_l0_small.load_state_dict(_c["state_dict"])
    sae_l0_small.eval()

_small_res = l0_small_sae_dir / "l0_small_correlation_results.pt"
if _small_res.exists() and 'l0s_features' not in dir():
    _r = torch.load(_small_res, weights_only=False)
    l0s_features = _r["features"]
    print(f"Loaded small L0 features from disk: {l0s_features.shape}")
else:
    sae_l0_small.eval()
    torch.cuda.empty_cache()
    
    N_EVAL = 10_000
    sample_l0s = l0_acts[:N_EVAL]
    recon_list, feat_list = [], []
    with torch.no_grad():
        for i in range(0, N_EVAL, SAE_BATCH_SIZE):
            batch = sample_l0s[i:i + SAE_BATCH_SIZE].to(device)
            out = sae_l0_small(batch)
            recon_list.append(out.reconstructed.cpu())
            feat_list.append(out.features.cpu())
            del out, batch
    
    recon_l0s = torch.cat(recon_list)
    l0s_features = torch.cat(feat_list).float()
    del recon_list, feat_list
    
    recon_mse = F.mse_loss(recon_l0s, sample_l0s).item()
    total_var = sample_l0s.var(dim=0).sum()
    fve = (1 - (sample_l0s - recon_l0s).var(dim=0).sum() / total_var).item()
    num_dead = (l0s_features.sum(dim=0) == 0).sum().item()
    l0s_sparsity = (l0s_features > 0).float().sum(dim=-1).mean().item()
    
    print(f"=== Small Layer 0 SAE (1×, k={L0_SMALL_K}) ===")
    print(f"Reconstruction MSE:              {recon_mse:.6f}")
    print(f"Fraction of variance explained:  {fve:.4f}")
    print(f"Dead features:                   {num_dead}/{l0_small_dict_size} ({100*num_dead/l0_small_dict_size:.1f}%)")
    print(f"L0 (avg active features):        {l0s_sparsity:.1f}")


### 7.3 Concept correlations

In [ ]:
concept_data = torch.load(
    RESULTS_DIR / "concept_labels.pt", map_location="cpu", weights_only=True
)
concept_labels_l0s = {k: v for k, v in concept_data["labels"].items() if k != "is_unvisited"}
concept_names_l0s = list(concept_labels_l0s.keys())

n = min(N_EVAL, min(v.shape[0] for v in concept_labels_l0s.values()))
labels_l0s = torch.stack([concept_labels_l0s[c][:n].float() for c in concept_names_l0s], dim=1)
feats_l0s = l0s_features[:n]

feat_c = feats_l0s - feats_l0s.mean(0, keepdim=True)
lab_c  = labels_l0s - labels_l0s.mean(0, keepdim=True)
feat_n = feat_c / feat_c.std(0, keepdim=True).clamp(min=1e-8)
lab_n  = lab_c  / lab_c.std(0,  keepdim=True).clamp(min=1e-8)
cm_l0s = (feat_n.T @ lab_n) / n
del feat_c, lab_c, feat_n, lab_n

MONO_THRESH = 0.4
mono_l0s = [
    i for i in range(l0_small_dict_size)
    if cm_l0s[i].abs().max() > MONO_THRESH and (cm_l0s[i].abs() > 0.3).sum() <= 2
]
dead_l0s = (feats_l0s.sum(dim=0) == 0).nonzero(as_tuple=True)[0].tolist()

print(f"=== Small Layer 0 SAE (1×, k={L0_SMALL_K}) vs 4× SAE ===")
print(f"Monosemantic (|r|>{MONO_THRESH}):  {len(mono_l0s)} / {l0_small_dict_size}")
print(f"Dead features:                      {len(dead_l0s)} / {l0_small_dict_size} ({100*len(dead_l0s)/l0_small_dict_size:.1f}%)")

if mono_l0s:
    print("\nMonosemantic features:")
    for fi in mono_l0s[:20]:
        corrs = {c: cm_l0s[fi, j].item() for j, c in enumerate(concept_names_l0s)}
        best = max(corrs, key=lambda k: abs(corrs[k]))
        print(f"  Feature {fi:4d} -> {best} (r={corrs[best]:+.3f})")

print("\nTop features per concept:")
for j, concept in enumerate(concept_names_l0s):
    top_idx = cm_l0s[:, j].abs().topk(min(5, l0_small_dict_size)).indices
    print(f"\n  {concept}:")
    for idx in top_idx:
        row = {c: cm_l0s[idx, k].item() for k, c in enumerate(concept_names_l0s)}
        print("    Feature {:4d}: ".format(idx.item()) +
              "  ".join(f"{c}={v:+.3f}" for c, v in row.items()))

torch.save(
    {"cm": cm_l0s, "concept_names": concept_names_l0s,
     "mono": mono_l0s, "dead": dead_l0s, "features": feats_l0s},
    l0_small_sae_dir / "l0_small_correlation_results.pt",
)
print(f"\nSaved to {l0_small_sae_dir / 'l0_small_correlation_results.pt'}")

---

## 8. Frontier-Specific Activation Analysis

`is_frontier` has ~6% prevalence, which caps Pearson r at ~0.24 — making it
hard to detect with standard correlation. This section uses two complementary
approaches:

1. **Feature enrichment on existing SAE** — which of the Layer 0 SAE's features
   fire *disproportionately* on frontier samples? We compute precision and
   enrichment ratio instead of Pearson r.

2. **Dedicated SAE on frontier activations only** — train a small SAE exclusively
   on (node, step) pairs where `is_frontier=1`. This removes the class-imbalance
   problem entirely and reveals what structure exists *within* frontier activations.

### 8.1 Filter frontier activations

In [ ]:
_front_path = RESULTS_DIR.parent.parent / "frontier_activations.pt"

if _front_path.exists() and 'l0_frontier_acts' not in dir():
    _fd = torch.load(_front_path, weights_only=False)
    l0_frontier_acts = _fd["l0_frontier_acts"]
    frontier_mask    = _fd["frontier_mask"]
    N_avail          = _fd["N_avail"]
    n_frontier       = frontier_mask.sum().item()
    print(f"Loaded frontier activations: {l0_frontier_acts.shape}")
    print(f"Frontier samples: {n_frontier:,} ({100*n_frontier/N_avail:.1f}%)")
else:
    concept_data = torch.load(
        RESULTS_DIR / "concept_labels.pt", map_location="cpu", weights_only=True
    )
    frontier_labels_all = concept_data["labels"]["is_frontier"]
    if 'per_layer_acts' not in dir() or not isinstance(per_layer_acts, dict):
        per_layer_acts = torch.load(RESULTS_DIR / "per_layer_activations.pt")
    l0_acts = per_layer_acts[0]
    N_avail       = min(frontier_labels_all.shape[0], l0_acts.shape[0])
    frontier_mask = frontier_labels_all[:N_avail].bool()
    n_frontier    = frontier_mask.sum().item()
    print(f"Total (node, step) samples: {N_avail:,}")
    print(f"Frontier samples:           {n_frontier:,} ({100*n_frontier/N_avail:.1f}%)")
    print(f"Expected max Pearson r:     {(n_frontier/N_avail * (1 - n_frontier/N_avail))**0.5:.3f}")
    l0_frontier_acts = l0_acts[:N_avail][frontier_mask]
    print(f"\nLayer 0 frontier activations: {l0_frontier_acts.shape}")
    torch.save({"l0_frontier_acts": l0_frontier_acts,
                "frontier_mask": frontier_mask, "N_avail": N_avail},
               _front_path)
    print(f"Saved to {_front_path}")

### 8.2 Enrichment analysis on existing Layer 0 SAE

In [ ]:
_enr_path = RESULTS_DIR.parent.parent / "frontier_enrichment.pt"
if _enr_path.exists() and 'enrichment' not in dir():
    _ed = torch.load(_enr_path, weights_only=False)
    enrichment = _ed["enrichment"]
    precision  = _ed["precision"]
    fr_f       = _ed["fr_frontier"]
    fr_nf      = _ed["fr_nonfrontier"]
    feat_eval  = _ed["feat_eval"]
    mask_eval  = _ed["mask_eval"]
    print(f"Loaded enrichment results from disk")
    # Still print the top features
    baseline_precision = mask_eval.float().mean().item()
    print(f"Frontier prevalence in eval set: {baseline_precision:.3f}")
    print(f"\nTop-10 features by enrichment:")
    top_enrich = enrichment.topk(10)
    for val, idx in zip(top_enrich.values, top_enrich.indices):
        fi = idx.item()
        prec = precision[fi].item()
        print(f"  Feature {fi:4d}: enrichment={val.item():.2f}×  precision={prec:.3f}")
else:
    # Reload Layer 0 SAE (4× version from Section 6) if not in memory
    if 'sae_l0' not in dir():
        l0_ckpt_path = RESULTS_DIR.parent.parent / "sae_layer0" / "sae_l0_final.pt"
        l0_ckpt = torch.load(l0_ckpt_path, map_location=device, weights_only=True)
        sae_l0 = BatchTopKSAE(
            input_dim=HIDDEN_DIM, dict_size=HIDDEN_DIM * 4, k=32
        ).to(device)
        sae_l0.load_state_dict(l0_ckpt["state_dict"])
        sae_l0.eval()
        l0_dict_size = HIDDEN_DIM * 4
        print("Reloaded Layer 0 SAE (4×, k=32)")
    
    # Ensure l0_dict_size is always defined
    if 'l0_dict_size' not in dir():
        l0_dict_size = HIDDEN_DIM * 4
    
    # Encode a balanced set: all frontier + equally many non-frontier
    N_EVAL_ENRICH = min(10_000, N_avail)
    mask_eval = frontier_mask[:N_EVAL_ENRICH]
    acts_eval = l0_acts[:N_avail][:N_EVAL_ENRICH]
    
    feat_eval_list = []
    with torch.no_grad():
        for i in range(0, N_EVAL_ENRICH, SAE_BATCH_SIZE):
            batch = acts_eval[i:i + SAE_BATCH_SIZE].to(device)
            out = sae_l0(batch)
            feat_eval_list.append(out.features.cpu())
            del out, batch
    feat_eval = torch.cat(feat_eval_list).float()  # (N_EVAL_ENRICH, dict_size)
    
    feat_frontier  = feat_eval[mask_eval]
    feat_nonfrontier = feat_eval[~mask_eval]
    
    # Firing rates
    fr_f  = (feat_frontier > 0).float().mean(0)     # (dict_size,)
    fr_nf = (feat_nonfrontier > 0).float().mean(0)
    
    # Enrichment: firing rate on frontier / firing rate off frontier
    enrichment = fr_f / (fr_nf + 1e-8)
    
    # Precision: among all times feature fires, what fraction are frontier?
    fired_total = (feat_eval > 0).float().sum(0)
    fired_on_frontier = (feat_frontier > 0).float().sum(0)
    precision = fired_on_frontier / (fired_total + 1e-8)
    
    # Baseline precision (random feature)
    baseline_precision = mask_eval.float().mean().item()
    print(f"Frontier prevalence in eval set: {baseline_precision:.3f} (baseline precision)")
    print(f"\nTop-10 features by enrichment (fires more on frontier):")
    top_enrich = enrichment.topk(10)
    for val, idx in zip(top_enrich.values, top_enrich.indices):
        fi = idx.item()
        prec = precision[fi].item()
        print(f"  Feature {fi:4d}: enrichment={val.item():.2f}×  precision={prec:.3f}  "
              f"fr(frontier)={fr_f[fi]:.4f}  fr(other)={fr_nf[fi]:.4f}")
    
    print(f"\nTop-10 features by precision:")
    top_prec = precision.topk(10)
    for val, idx in zip(top_prec.values, top_prec.indices):
        fi = idx.item()
        enr = enrichment[fi].item()
        print(f"  Feature {fi:4d}: precision={val.item():.3f}  enrichment={enr:.2f}×  "
              f"fr(frontier)={fr_f[fi]:.4f}")
# Save enrichment results immediately
_enr_path = RESULTS_DIR.parent.parent / "frontier_enrichment.pt"
torch.save({"enrichment": enrichment, "precision": precision,
            "fr_frontier": fr_f, "fr_nonfrontier": fr_nf,
            "feat_eval": feat_eval, "mask_eval": mask_eval},
           _enr_path)
print(f"Saved enrichment results to {_enr_path}")


### 8.3 Train dedicated SAE on frontier activations only

In [ ]:
FRONTIER_EXPANSION = 2
FRONTIER_K         = 2 if LOCAL_DEBUG else 4
frontier_dict_size = HIDDEN_DIM * FRONTIER_EXPANSION
frontier_sae_dir   = RESULTS_DIR.parent.parent / "sae_frontier"
frontier_sae_dir.mkdir(parents=True, exist_ok=True)
_front_sae_ckpt = frontier_sae_dir / "sae_frontier_final.pt"

sae_frontier = BatchTopKSAE(
    input_dim=HIDDEN_DIM, dict_size=frontier_dict_size, k=FRONTIER_K
).to(device)

if _front_sae_ckpt.exists():
    _c = torch.load(_front_sae_ckpt, map_location=device, weights_only=True)
    sae_frontier.load_state_dict(_c["state_dict"])
    sae_frontier.eval()
    print(f"Loaded frontier SAE from {_front_sae_ckpt.name}")
else:
    print(f"Training frontier SAE on {l0_frontier_acts.shape[0]} samples")
    trainer_frontier = SAETrainer(sae_frontier, lr=SAE_LR, resample_dead_every=0)
    frontier_loader  = make_activation_dataloader(
        l0_frontier_acts,
        batch_size=min(SAE_BATCH_SIZE, max(32, len(l0_frontier_acts) // 8)),
    )
    FRONTIER_STEPS = max(SAE_STEPS // 4, 5_000)
    step = 0
    while step < FRONTIER_STEPS:
        for (batch,) in frontier_loader:
            if step >= FRONTIER_STEPS:
                break
            output = trainer_frontier.train_step(batch.to(device))
            if step % 2_000 == 0:
                print(f"Step {step}/{FRONTIER_STEPS} | loss={output.loss.item():.4f} | L0={output.l0.item():.1f}")
            step += 1
    torch.save({"state_dict": sae_frontier.state_dict(), "config": sae_frontier.get_config()},
               _front_sae_ckpt)
    print(f"Saved to {_front_sae_ckpt}")

### 8.4 Evaluate frontier SAE — what varies *within* frontier moments?

In [ ]:
sae_frontier.eval()
torch.cuda.empty_cache()

n_eval_f = min(len(l0_frontier_acts), 5_000)
acts_f_eval = l0_frontier_acts[:n_eval_f]

# Encode frontier samples
feat_f_list, recon_f_list = [], []
with torch.no_grad():
    for i in range(0, n_eval_f, SAE_BATCH_SIZE):
        batch = acts_f_eval[i:i + SAE_BATCH_SIZE].to(device)
        out = sae_frontier(batch)
        feat_f_list.append(out.features.cpu())
        recon_f_list.append(out.reconstructed.cpu())
        del out, batch

feat_f = torch.cat(feat_f_list).float()
recon_f = torch.cat(recon_f_list)
del feat_f_list, recon_f_list

fve_f = (1 - (acts_f_eval - recon_f).var(dim=0).sum() / acts_f_eval.var(dim=0).sum()).item()
num_dead_f = (feat_f.sum(0) == 0).sum().item()

print(f"Frontier SAE (2×, k={FRONTIER_K})")
print(f"FVE: {fve_f:.4f}")
print(f"Dead features: {num_dead_f}/{frontier_dict_size} ({100*num_dead_f/frontier_dict_size:.1f}%)")

# Concept correlations within frontier samples
# is_frontier is constant=1 here; is_source and is_visited still vary
frontier_concept_labels = {
    c: concept_data["labels"][c][:N_avail][frontier_mask][:n_eval_f].float()
    for c in ["is_source", "is_visited"]
}
concept_names_f = list(frontier_concept_labels.keys())
labels_f = torch.stack([frontier_concept_labels[c] for c in concept_names_f], dim=1)

feat_c = feat_f - feat_f.mean(0, keepdim=True)
lab_c  = labels_f - labels_f.mean(0, keepdim=True)
feat_n = feat_c / feat_c.std(0, keepdim=True).clamp(min=1e-8)
lab_n  = lab_c  / lab_c.std(0,  keepdim=True).clamp(min=1e-8)
cm_f = (feat_n.T @ lab_n) / feat_f.shape[0]
del feat_c, lab_c, feat_n, lab_n

print(f"\nWithin-frontier concept correlations:")
print(f"  (is_frontier=1 for all samples; is_source and is_visited vary)")
for j, concept in enumerate(concept_names_f):
    top_idx = cm_f[:, j].abs().topk(min(5, frontier_dict_size)).indices
    print(f"\n  {concept}:")
    for idx in top_idx:
        row = {c: cm_f[idx, k].item() for k, c in enumerate(concept_names_f)}
        print("    Feature {:3d}: ".format(idx.item()) +
              "  ".join(f"{c}={v:+.3f}" for c, v in row.items()))

torch.save(
    {"cm": cm_f, "concept_names": concept_names_f, "features": feat_f,
     "enrichment": enrichment, "precision": precision,
     "fr_frontier": fr_f, "fr_nonfrontier": fr_nf},
    frontier_sae_dir / "frontier_analysis_results.pt",
)
print(f"\nSaved to {frontier_sae_dir / 'frontier_analysis_results.pt'}")

---

## 9. Linear Probe Baseline

A linear probe fits logistic regression directly on raw activations (no SAE) per
concept per layer. This answers: *"Is the concept linearly decodable from the
activations at all?"*

This sets the ceiling for what the SAE should be finding:
- If probe AUROC ≈ 1.0 for `is_frontier` at Layer 0 but SAE r ≈ 0.24 → SAE is underperforming.
- If probe AUROC ≈ 0.55 → concept is barely encoded, weak SAE r is expected.

We use **AUROC** as the primary metric — unlike accuracy or Pearson r, AUROC is
not affected by class imbalance (6% frontier still gets a fair score).

### 9.1 Fit probes per layer and concept

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import train_test_split
import numpy as np

# Load activations and labels
if 'per_layer_acts' not in dir() or not isinstance(per_layer_acts, dict):
    per_layer_acts = torch.load(RESULTS_DIR / "per_layer_activations.pt")

concept_data = torch.load(
    RESULTS_DIR / "concept_labels.pt", map_location="cpu", weights_only=True
)
probe_concepts = ["is_source", "is_visited", "is_frontier"]

N_PROBE = 20_000  # subsample for speed; stratified split preserves class ratios

probe_results = {}  # (layer_idx, concept) -> {"auroc": float, "f1": float}

for layer_idx in range(NUM_LAYERS):
    acts = per_layer_acts[layer_idx]
    N = min(N_PROBE, acts.shape[0])
    X = acts[:N].numpy().astype(np.float32)

    for concept in probe_concepts:
        y = concept_data["labels"][concept][:N].numpy().astype(np.int32)

        pos_rate = y.mean()
        if pos_rate < 0.001 or pos_rate > 0.999:
            print(f"L{layer_idx} | {concept:12s} | skipped (prevalence={pos_rate:.4f})")
            continue

        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=y
        )

        clf = LogisticRegression(
            class_weight="balanced", max_iter=1000, solver="lbfgs", C=1.0, n_jobs=-1
        )
        clf.fit(X_tr, y_tr)

        y_prob = clf.predict_proba(X_te)[:, 1]
        y_pred = clf.predict(X_te)

        auroc = roc_auc_score(y_te, y_prob)
        f1   = f1_score(y_te, y_pred, zero_division=0)

        probe_results[(layer_idx, concept)] = {"auroc": auroc, "f1": f1, "clf": clf}
        print(f"L{layer_idx} | {concept:12s} | AUROC={auroc:.3f} | F1={f1:.3f} | "
              f"prevalence={pos_rate:.3f}")
# Save probe results immediately after fitting
_probe_save = {str(k): {"auroc": v["auroc"], "f1": v["f1"]}
               for k, v in probe_results.items()}
_probe_path = RESULTS_DIR.parent.parent / "linear_probes"
_probe_path.mkdir(parents=True, exist_ok=True)
torch.save({"results": _probe_save, "concepts": probe_concepts, "num_layers": NUM_LAYERS},
           _probe_path / "probe_results.pt")
print(f"Saved probe results to {_probe_path / 'probe_results.pt'}")

### 9.2 Results table

In [ ]:
# Print AUROC table: rows=concepts, cols=layers
print("AUROC by layer and concept")
print("-" * (16 + 10 * NUM_LAYERS))
header = f"{'Concept':14s}" + "".join(f"  Layer {i}" for i in range(NUM_LAYERS))
print(header)
print("-" * len(header))
for concept in probe_concepts:
    row = f"{concept:14s}"
    for layer_idx in range(NUM_LAYERS):
        key = (layer_idx, concept)
        if key in probe_results:
            row += f"  {probe_results[key]['auroc']:.3f}  "
        else:
            row += "   N/A   "
    print(row)

print()
print("F1 by layer and concept (class_weight='balanced')")
print("-" * len(header))
print(header)
print("-" * len(header))
for concept in probe_concepts:
    row = f"{concept:14s}"
    for layer_idx in range(NUM_LAYERS):
        key = (layer_idx, concept)
        if key in probe_results:
            row += f"  {probe_results[key]['f1']:.3f}  "
        else:
            row += "   N/A   "
    print(row)

### 9.3 Heatmap + comparison to SAE

In [ ]:
if 'probe_results' not in dir():
    _pr = torch.load(RESULTS_DIR.parent.parent / "linear_probes" / "probe_results.pt",
                     weights_only=False)
    # Restore dict with tuple keys
    probe_results   = {eval(k): v for k, v in _pr["results"].items()}
    probe_concepts  = _pr["concepts"]
    print("Loaded probe results from disk")

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

auroc_matrix = np.full((len(probe_concepts), NUM_LAYERS), np.nan)
f1_matrix    = np.full((len(probe_concepts), NUM_LAYERS), np.nan)

for i, concept in enumerate(probe_concepts):
    for j in range(NUM_LAYERS):
        key = (j, concept)
        if key in probe_results:
            auroc_matrix[i, j] = probe_results[key]["auroc"]
            f1_matrix[i, j]    = probe_results[key]["f1"]

layer_labels = [f"L{i}" for i in range(NUM_LAYERS)]

for ax, mat, title in [
    (axes[0], auroc_matrix, "Linear Probe AUROC"),
    (axes[1], f1_matrix,    "Linear Probe F1"),
]:
    im = ax.imshow(mat, vmin=0.5, vmax=1.0, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(NUM_LAYERS))
    ax.set_xticklabels(layer_labels)
    ax.set_yticks(range(len(probe_concepts)))
    ax.set_yticklabels(probe_concepts)
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
    for i in range(len(probe_concepts)):
        for j in range(NUM_LAYERS):
            if not np.isnan(mat[i, j]):
                ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center",
                        fontsize=9, color="black")

plt.tight_layout()
results_dir_probe = RESULTS_DIR.parent.parent / "linear_probes"
results_dir_probe.mkdir(parents=True, exist_ok=True)
plt.savefig(results_dir_probe / "probe_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# Save probe results (without clf objects)
probe_save = {
    str(k): {"auroc": v["auroc"], "f1": v["f1"]}
    for k, v in probe_results.items()
}
torch.save({"results": probe_save, "concepts": probe_concepts,
            "num_layers": NUM_LAYERS},
           results_dir_probe / "probe_results.pt")
print(f"Saved to {results_dir_probe / 'probe_results.pt'}")


---

## 10. Per-Step Temporal Analysis

All previous sections flatten every `(node, step)` pair together — losing the
temporal structure of BFS execution. This section restores it:

1. Run a single batch through the model with per-step activation collection.
2. Extract ground-truth BFS state at each step from hints (`reach_h`).
3. Encode each step's Layer 0 activations with the SAE.
4. Visualise **feature trajectories**: does each feature track the BFS wavefront?

### 10.1 Run one batch, extract per-step Layer 0 activations

In [ ]:
_psa_path = RESULTS_DIR.parent.parent / "per_step_analysis.pt"

if _psa_path.exists() and 'step_acts_g' not in dir():
    _psd = torch.load(_psa_path, weights_only=False)
    step_acts_g = _psd["step_acts_g"]
    n_nodes     = _psd["n_nodes"]
    n_steps     = _psd["n_steps"]
    g           = _psd["g"]
    max_steps   = _psd["max_steps"]
    print(f"Loaded per-step activations from disk: {step_acts_g.shape}")
    print(f"Graph {g}: {n_nodes} nodes, {n_steps} BFS steps")
else:
    # Re-create a small val loader if needed (same seed as Section 2.2)
    from data.clrs_dataset import get_clrs_dataloader
    
    _step_loader = get_clrs_dataloader(
        ALGORITHM, "val",
        batch_size=16,          # small batch — we visualise one graph
        num_samples=64,
        num_nodes=NUM_NODES,
        edge_probability=EDGE_PROB,
        seed=SEED,
        num_workers=0,
    )
    single_batch = next(iter(_step_loader))
    inputs, _, hints = batch_to_model_inputs(single_batch, spec)   # hints on CPU
    inputs_dev = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                  for k, v in inputs.items()}
    max_steps = single_batch.lengths.max().item()
    
    model.eval()
    with torch.no_grad():
        output = model(
            inputs=inputs_dev,
            output_types={},
            num_steps=max_steps,
            return_activations=True,
        )
    
    # layer_activations[0] is a list of dicts, one per model step
    # Each dict has 'node_mlp_output': (batch, max_nodes, hidden_dim)
    layer0_step_dicts = output.activations['layer_activations'][0]
    print(f"Batch: {single_batch.num_graphs} graphs, max_steps={max_steps}")
    print(f"Layer 0 step dicts: {len(layer0_step_dicts)}")
    
    # Pick the graph with the most steps (most interesting BFS trace)
    g = single_batch.lengths.argmax().item()
    n_nodes = (single_batch.batch == g).sum().item()
    n_steps = single_batch.lengths[g].item()
    print(f"Visualising graph {g}: {n_nodes} nodes, {n_steps} BFS steps")
    
    # Stack per-step Layer 0 activations for graph g: (max_steps, n_nodes, hidden_dim)
    step_acts_g = torch.stack([
        d['node_mlp_output'][g, :n_nodes].cpu()
        for d in layer0_step_dicts
    ])  # (max_steps, n_nodes, hidden_dim)
    print(f"Per-step activations shape: {step_acts_g.shape}")
    torch.save({"step_acts_g": step_acts_g, "n_nodes": n_nodes,
                "n_steps": n_steps, "g": g, "max_steps": max_steps},
               _psa_path)
    print(f"Saved per-step activations to {_psa_path}")


### 10.2 Extract BFS ground-truth labels per step

In [ ]:
_psa_path = RESULTS_DIR.parent.parent / "per_step_analysis.pt"

if _psa_path.exists() and 'visited' not in dir():
    _psd = torch.load(_psa_path, weights_only=False)
    if "visited" in _psd:
        visited     = _psd["visited"]
        frontier    = _psd["frontier"]
        source_node = _psd["source_node"]
        print(f"Loaded BFS labels from disk: {visited.shape}")
    else:
        # activations saved but labels not yet — fall through to compute
        pass

if 'visited' not in dir():
    if 'hints' not in dir():
        raise RuntimeError("hints not in memory — re-run Section 10.1 cell first")
    # hints['reach_h']: (batch, max_nodes, max_steps)  — is_visited at each step
    reach_h = hints.get('reach_h')
    if reach_h is None:
        raise RuntimeError("reach_h not found in hints. Check that ALGORITHM='bfs'.")
    
    visited  = reach_h[g, :n_nodes, :n_steps].float()    # (n_nodes, n_steps)
    frontier = torch.zeros_like(visited)
    frontier[:, 0] = visited[:, 0]
    frontier[:, 1:] = visited[:, 1:] * (1 - visited[:, :-1])
    
    # Source node (static)
    source_node = visited[:, 0].argmax().item() if visited[:, 0].sum() > 0 else 0
    
    print(f"Visited matrix: {visited.shape}  (nodes × steps)")
    print(f"Total frontier events: {frontier.sum().int().item()}")
    print(f"Source node: {source_node}")
    # Save labels back into the same file alongside activations
    _update = torch.load(_psa_path, weights_only=False) if _psa_path.exists() else {}
    _update.update({"visited": visited, "frontier": frontier, "source_node": source_node})
    torch.save(_update, _psa_path)
    print(f"Saved BFS labels to {_psa_path}")


### 10.3 Encode each step with Layer 0 SAE

In [ ]:
_psa_path = RESULTS_DIR.parent.parent / "per_step_analysis.pt"

if _psa_path.exists() and 'step_feats_g' not in dir():
    _psd = torch.load(_psa_path, weights_only=False)
    if "step_feats_g" in _psd:
        step_feats_g     = _psd["step_feats_g"]
        mono_l0          = _psd.get("mono_l0", [])
        cm_l0            = _psd.get("cm_l0")
        concept_names_l0 = _psd.get("concept_names_l0", [])
        top_feats        = _psd.get("top_feats", [])
        l0_dict_size     = step_feats_g.shape[-1]
        print(f"Loaded SAE features from disk: {step_feats_g.shape}")

if 'step_feats_g' not in dir():
    # Load Layer 0 SAE if not already in memory
    if 'sae_l0' not in dir():
        _l0_ckpt = torch.load(
            RESULTS_DIR.parent.parent / "sae_layer0" / "sae_l0_final.pt",
            map_location=device, weights_only=True,
        )
        sae_l0 = BatchTopKSAE(input_dim=HIDDEN_DIM, dict_size=HIDDEN_DIM * 4, k=32).to(device)
        sae_l0.load_state_dict(_l0_ckpt["state_dict"])
        sae_l0.eval()
        l0_dict_size = HIDDEN_DIM * 4
        print("Reloaded Layer 0 SAE (4×, k=32)")
    
    # Encode each step: (max_steps, n_nodes, dict_size)
    sae_l0.eval()
    step_feats_g = []
    with torch.no_grad():
        for t in range(max_steps):
            batch_t = step_acts_g[t].to(device)  # (n_nodes, hidden_dim)
            out_t = sae_l0(batch_t)
            step_feats_g.append(out_t.features.cpu())
            del out_t, batch_t
    
    step_feats_g = torch.stack(step_feats_g)  # (max_steps, n_nodes, dict_size)
    print(f"SAE features per step: {step_feats_g.shape}")
    
    # Load monosemantic feature list
    if 'mono_l0' not in dir() or 'cm_l0' not in dir():
        _res = torch.load(
            RESULTS_DIR.parent.parent / "sae_layer0" / "l0_correlation_results.pt"
        )
        mono_l0 = _res["mono"]
        cm_l0   = _res["cm"]
        concept_names_l0 = _res["concept_names"]
        l0_dict_size = cm_l0.shape[0]
    
    # Features to visualise: monosemantic ones, or top-4 by |r(is_visited)|
    vis_concept = "is_visited"
    if concept_names_l0 and vis_concept in concept_names_l0:
        j_vis = concept_names_l0.index(vis_concept)
        top_feats = cm_l0[:, j_vis].abs().topk(4).indices.tolist()
    else:
        top_feats = mono_l0[:4] if mono_l0 else list(range(4))
    
    print(f"Features to plot: {top_feats}")
    # Save into per_step_analysis.pt
    _update = torch.load(_psa_path, weights_only=False) if _psa_path.exists() else {}
    _update.update({"step_feats_g": step_feats_g, "top_feats": top_feats,
                    "mono_l0": mono_l0 if 'mono_l0' in dir() else [],
                    "cm_l0": cm_l0 if 'cm_l0' in dir() else None,
                    "concept_names_l0": concept_names_l0 if 'concept_names_l0' in dir() else []})
    torch.save(_update, _psa_path)
    print(f"Saved SAE features to {_psa_path}")


### 10.4 Visualise: feature trajectories vs BFS wavefront

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

n_feat_rows = len(top_feats)
fig = plt.figure(figsize=(max(10, n_steps * 0.9), 3 * (2 + n_feat_rows)))
gs = gridspec.GridSpec(2 + n_feat_rows, 1, hspace=0.45)

_extent = [-0.5, n_steps - 0.5, n_nodes - 0.5, -0.5]

# ── Row 0: is_visited (ground truth) ──────────────────────────────────────────
ax0 = fig.add_subplot(gs[0])
im0 = ax0.imshow(visited.numpy(), aspect='auto', cmap='Blues', vmin=0, vmax=1,
                  extent=_extent, interpolation='nearest')
ax0.axhline(source_node - 0.5, color='red', lw=1.5, linestyle='--', alpha=0.7)
ax0.axhline(source_node + 0.5, color='red', lw=1.5, linestyle='--', alpha=0.7)
ax0.set_ylabel("Node")
ax0.set_title("Ground truth: is_visited  (red dashes = source node)")
plt.colorbar(im0, ax=ax0, fraction=0.02)

# ── Row 1: is_frontier (ground truth) ─────────────────────────────────────────
ax1 = fig.add_subplot(gs[1])
im1 = ax1.imshow(frontier.numpy(), aspect='auto', cmap='Oranges', vmin=0, vmax=1,
                  extent=_extent, interpolation='nearest')
ax1.set_ylabel("Node")
ax1.set_title("Ground truth: is_frontier  (node first enters BFS wavefront)")
plt.colorbar(im1, ax=ax1, fraction=0.02)

# ── Rows 2+: SAE feature activations ──────────────────────────────────────────
for k, feat_idx in enumerate(top_feats):
    ax = fig.add_subplot(gs[2 + k])
    # feat_acts: (n_steps, n_nodes) → transpose to (n_nodes, n_steps)
    feat_acts = step_feats_g[:n_steps, :, feat_idx].numpy().T  # (n_nodes, n_steps)
    im = ax.imshow(feat_acts, aspect='auto', cmap='Reds', vmin=0,
                   extent=_extent, interpolation='nearest')
    if concept_names_l0:
        corr_str = "  ".join(
            f"{c}={cm_l0[feat_idx, j].item():+.3f}" for j, c in enumerate(concept_names_l0)
        )
    else:
        corr_str = ""
    ax.set_ylabel("Node")
    ax.set_title(f"Feature {feat_idx} activation  [{corr_str}]")
    plt.colorbar(im, ax=ax, fraction=0.02)
    if k == n_feat_rows - 1:
        ax.set_xlabel("BFS step")

_save_dir = RESULTS_DIR.parent.parent / "sae_layer0"
_save_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(_save_dir / "per_step_feature_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {_save_dir / 'per_step_feature_trajectories.png'}")

### 10.5 Line-plot: activation trajectory for the top feature

In [ ]:
# For the single strongest feature, plot each node's activation as a line over time.
# Nodes are coloured by when they became visited: source=red, early=dark blue, late=light blue.
top_feat = top_feats[0] if top_feats else 0
feat_traj = step_feats_g[:n_steps, :, top_feat].numpy()  # (n_steps, n_nodes)

# Colour each node by its first-visited step (earlier = darker)
first_visited = []
for n in range(n_nodes):
    v_steps = visited[n, :].nonzero(as_tuple=False)
    first_visited.append(v_steps[0].item() if len(v_steps) > 0 else n_steps)

cmap = plt.cm.Blues
norm_fv = plt.Normalize(vmin=0, vmax=max(first_visited) + 1)

fig, ax = plt.subplots(figsize=(max(8, n_steps * 0.7), 4))
for n in range(n_nodes):
    color = 'red' if n == source_node else cmap(norm_fv(first_visited[n]))
    lw = 2.0 if n == source_node else 0.9
    label = f"node {n} (src)" if n == source_node else None
    ax.plot(range(n_steps), feat_traj[:, n], color=color, lw=lw,
            alpha=0.85, label=label)

# Mark frontier steps for each node
for n in range(n_nodes):
    for t in range(n_steps):
        if frontier[n, t] > 0:
            ax.axvline(t, color='orange', alpha=0.25, lw=0.8)

ax.set_xlabel("BFS step")
ax.set_ylabel("Feature activation")
corr_str = ""
if concept_names_l0:
    corr_str = "  ".join(f"{c}={cm_l0[top_feat, j].item():+.3f}"
                         for j, c in enumerate(concept_names_l0))
ax.set_title(f"Feature {top_feat} trajectory per node\n[{corr_str}]\n"
             f"(orange lines = frontier events, red = source node)")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig(_save_dir / "per_step_top_feature_lines.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {_save_dir / 'per_step_top_feature_lines.png'}")

---

## 11. Cross-Section Summary

Combines results from all sections into one table and figure:
- **Section 6**: SAE max |r| and monosemantic count per layer
- **Section 9**: Linear probe AUROC per layer  
- Highlights where the SAE is close to the probe ceiling and where it is not.

### 11.1 Load all saved results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

_run_root = RESULTS_DIR.parent.parent

# ── Layer SAE results ─────────────────────────────────────────────────────────
if 'layer_eval' not in dir():
    layer_eval = {}
    for li in range(NUM_LAYERS):
        p = _run_root / f"sae_layer{li}" / f"l{li}_correlation_results.pt"
        if p.exists():
            layer_eval[li] = torch.load(p, weights_only=False)
            print(f"Loaded L{li} SAE results")
        else:
            print(f"[missing] {p} — run Section 6 first")

if 'concept_names' not in dir() and layer_eval:
    concept_names = layer_eval[0]["concept_names"]

# ── Probe results ─────────────────────────────────────────────────────────────
_probe_path = _run_root / "linear_probes" / "probe_results.pt"
probe_data  = None
if _probe_path.exists():
    probe_data = torch.load(_probe_path, weights_only=False)
    print("Loaded linear probe results")
else:
    print(f"[missing] {_probe_path} — run Section 9 to get probe baseline")

# ── Small L0 SAE results ──────────────────────────────────────────────────────
_small_path = _run_root / "sae_layer0_small" / "l0_small_correlation_results.pt"
small_l0_data = None
if _small_path.exists():
    small_l0_data = torch.load(_small_path, weights_only=False)
    print("Loaded small L0 SAE results")

# ── LAYER_K fallback (defined in Section 6 training cell; guard for standalone runs) ─
if 'LAYER_K' not in dir():
    LAYER_K = 16 if LOCAL_DEBUG else 32

print(f"\nAvailable layers: {sorted(layer_eval.keys())}")

### 11.2 Consolidated summary table

In [ ]:
_concepts = concept_names if 'concept_names' in dir() else []
_layers   = sorted(layer_eval.keys()) if layer_eval else []

if not _layers or not _concepts:
    print("Nothing to display — run Sections 6 and 9 first.")
else:
    MONO_T = 0.4
    _cw = 8

    # ── Header ────────────────────────────────────────────────────────────────
    print("\n" + "="*80)
    print("CROSS-SECTION SUMMARY")
    print("="*80)

    # ── SAE results ───────────────────────────────────────────────────────────
    print("\n── SAE (4×, k=32) ── max |r| per concept")
    _h = f"{'':14s}" + "".join(f"{'L'+str(li):>{_cw}}" for li in _layers)
    print(_h); print("─"*len(_h))
    for j, c in enumerate(_concepts):
        row = f"{c:14s}"
        for li in _layers:
            if li in layer_eval:
                row += f"{layer_eval[li]['cm'][:, j].abs().max().item():>{_cw}.3f}"
            else:
                row += f"{'N/A':>{_cw}}"
        print(row)
    print("─"*len(_h))
    print(f"{'mono (≥0.4)':14s}" +
          "".join(f"{len(layer_eval[li]['mono']) if li in layer_eval else 0:>{_cw}}"
                  for li in _layers))
    print(f"{'dead %':14s}" +
          "".join(f"{layer_eval[li]['dead_pct']:>{_cw}.1f}" if li in layer_eval else f"{'N/A':>{_cw}}"
                  for li in _layers))

    # ── Probe AUROC ───────────────────────────────────────────────────────────
    if probe_data:
        print("\n── Linear Probe ── AUROC per concept")
        print(_h); print("─"*len(_h))
        for c in _concepts:
            row = f"{c:14s}"
            for li in _layers:
                key = str((li, c))
                if key in probe_data["results"]:
                    row += f"{probe_data['results'][key]['auroc']:>{_cw}.3f}"
                else:
                    row += f"{'N/A':>{_cw}}"
            print(row)

    # ── SAE vs Probe gap ──────────────────────────────────────────────────────
    if probe_data:
        print("\n── Gap: AUROC − scaled max|r|  (negative = SAE underperforms probe)")
        print("   (Pearson r is not directly comparable to AUROC; shown for reference)")
        print(_h); print("─"*len(_h))
        for j, c in enumerate(_concepts):
            row = f"{c:14s}"
            for li in _layers:
                key = str((li, c))
                if key in probe_data["results"] and li in layer_eval:
                    auroc   = probe_data["results"][key]["auroc"]
                    max_r   = layer_eval[li]["cm"][:, j].abs().max().item()
                    # normalise r to [0.5, 1] range for rough comparability
                    r_scaled = 0.5 + max_r / 2
                    gap = auroc - r_scaled
                    row += f"{gap:>+{_cw}.3f}"
                else:
                    row += f"{'N/A':>{_cw}}"
            print(row)
        print("  Gap > 0 → probe finds information the SAE is missing")

    # ── Small L0 SAE comparison ───────────────────────────────────────────────
    if small_l0_data:
        print("\n── Small L0 SAE (1×, k=8) vs 4× SAE ──")
        print(f"  Mono (1×): {len(small_l0_data['mono'])}  |  "
              f"Mono (4×): {len(layer_eval[0]['mono']) if 0 in layer_eval else 'N/A'}")
        print(f"  Dead (1×): {len(small_l0_data['dead'])/len(small_l0_data['dead'])*100:.0f}%  |  "
              f"Dead (4×): {layer_eval[0]['dead_pct'] if 0 in layer_eval else 'N/A':.1f}%")
        for j, c in enumerate(small_l0_data["concept_names"]):
            best_small = small_l0_data["cm"][:, j].abs().max().item()
            best_large = layer_eval[0]["cm"][:, j].abs().max().item() if 0 in layer_eval else float('nan')
            print(f"  {c:14s}: 1× max|r|={best_small:.3f}  4× max|r|={best_large:.3f}")

    print("\n" + "="*80)

### 11.3 Combined heatmap: SAE max |r| vs probe AUROC

In [ ]:
if not layer_eval or not concept_names:
    print("Run Section 6 first.")
else:
    n_concepts = len(concept_names)
    n_layers   = NUM_LAYERS
    has_probe  = probe_data is not None

    n_cols  = 2 if has_probe else 1
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols * n_layers / 4 + 2,
                                                  max(3, n_concepts * 0.8)))

    sae_mat = np.array([
        [layer_eval[li]["cm"][:, j].abs().max().item() if li in layer_eval else np.nan
         for li in range(n_layers)]
        for j in range(n_concepts)
    ])

    ax_sae = axes[0] if has_probe else axes
    im1 = ax_sae.imshow(sae_mat, vmin=0, vmax=1, cmap="YlOrRd", aspect="auto")
    ax_sae.set_xticks(range(n_layers))
    ax_sae.set_xticklabels([f"L{i}" for i in range(n_layers)])
    ax_sae.set_yticks(range(n_concepts))
    ax_sae.set_yticklabels(concept_names)
    ax_sae.set_title(f"SAE max |Pearson r|\n(4×, k={LAYER_K})")
    plt.colorbar(im1, ax=ax_sae, fraction=0.04)
    for i in range(n_concepts):
        for j in range(n_layers):
            if not np.isnan(sae_mat[i, j]):
                ax_sae.text(j, i, f"{sae_mat[i,j]:.2f}",
                            ha="center", va="center", fontsize=8)

    if has_probe:
        probe_mat = np.array([
            [probe_data["results"].get(str((li, c)), {}).get("auroc", np.nan)
             for li in range(n_layers)]
            for c in concept_names
        ])
        ax_pr = axes[1]
        im2 = ax_pr.imshow(probe_mat, vmin=0.5, vmax=1, cmap="YlGn", aspect="auto")
        ax_pr.set_xticks(range(n_layers))
        ax_pr.set_xticklabels([f"L{i}" for i in range(n_layers)])
        ax_pr.set_yticks(range(n_concepts))
        ax_pr.set_yticklabels(concept_names)
        ax_pr.set_title("Linear Probe AUROC\n(logistic regression, class_weight=balanced)")
        plt.colorbar(im2, ax=ax_pr, fraction=0.04)
        for i in range(n_concepts):
            for j in range(n_layers):
                if not np.isnan(probe_mat[i, j]):
                    ax_pr.text(j, i, f"{probe_mat[i,j]:.2f}",
                               ha="center", va="center", fontsize=8)

    plt.suptitle("SAE features vs linear probe baseline — BFS concepts by layer",
                 fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig(_run_root / "summary_sae_vs_probe.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {_run_root / 'summary_sae_vs_probe.png'}")